In [ ]:
import os
import sys
import math
import time
import random
import warnings
import subprocess
from pathlib import Path
from dataclasses import dataclass

warnings.filterwarnings("ignore")

def pip_install(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return True
    except Exception as e:
        print(f"[WARN] Could not install {pkg}: {e}")
        return False

try:
    import einops
except Exception:
    pip_install("einops")

try:
    import timm
except Exception:
    pip_install("timm")

# Optional IQA metrics
PYIQA_AVAILABLE = False
try:
    import pyiqa
    PYIQA_AVAILABLE = True
except Exception:
    ok = pip_install("pyiqa")
    if ok:
        try:
            import pyiqa
            PYIQA_AVAILABLE = True
        except Exception:
            PYIQA_AVAILABLE = False

try:
    import thop
    THOP_AVAILABLE = True
except Exception:
    ok = pip_install("thop")
    if ok:
        try:
            import thop
            THOP_AVAILABLE = True
        except Exception:
            THOP_AVAILABLE = False
    else:
        THOP_AVAILABLE = False


In [ ]:
# 1. Imports and configuration
# ============================================================

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from torchvision.transforms import functional as TF

from einops import rearrange

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import torch

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

In [ ]:
# Main experiment switches
# ----------------------------
from dataclasses import dataclass
import os
import random
import numpy as np
import torch

@dataclass
class CFG:
    # ============================
    # FINAL FAIR COMPARISON MODE: UAVid C=128
    # ============================
    SMOKE_TEST: bool = False

    # ============================
    # Training settings
    # Base paper: 500 epochs, batch size 16, Adam, LR=1e-4
    # Kaggle T4 uses batch 8 + accumulation 2 => effective batch 16.
    # ============================
    EPOCHS: int = 500
    BATCH_SIZE: int = 8
    ACCUM_STEPS: int = 2   # effective batch size = 16

    LR: float = 1e-4
    MIN_LR: float = 1e-6
    WEIGHT_DECAY: float = 0.0

    # ============================
    # Super-resolution setting
    # UAVid base-paper pair:
    # HR 512x512 -> LR 128x128
    # Training crop:
    # HR 128x128 -> LR 32x32
    # ============================
    SCALE: int = 4
    HR_CROP: int = 512
    TRAIN_HR_CROP: int = 128

    # ============================
    # Model settings
    # C=128 matches the full LSTMConvSR width in the base paper.
    # ============================
    EMBED_DIM: int = 128
    NUM_BLOCKS: int = 4
    FAMCB_PER_GROUP: int = 4
    MLP_RATIO: float = 2.0

    # ============================
    # Runtime settings
    # ============================
    NUM_WORKERS: int = 4
    SEED: int = 42

    # ============================
    # Dataset selection: UAVid only
    # ============================
    USE_POTSDAM: bool = False
    USE_UAVID: bool = True
    USE_RSSCN7: bool = False

    # Smoke-test limits, unused when SMOKE_TEST=False
    MAX_TRAIN_SAMPLES_SMOKE: int = 300
    MAX_VAL_SAMPLES_SMOKE: int = 80
    MAX_TEST_SAMPLES_SMOKE: int = 80

    # Full-run limits
    MAX_TRAIN_SAMPLES_FULL = None
    MAX_VAL_SAMPLES_FULL = None
    MAX_TEST_SAMPLES_FULL = None

    # Loss weights
    LAMBDA_FREQ: float = 0.05
    LAMBDA_EDGE: float = 0.02
    LAMBDA_PER: float = 0.0

    # ============================
    # Base-paper UAVid scalar baselines for one-sample statistical comparison.
    # Full LSTMConvSR reported values on UAVid from Table 6:
    # PSNR=28.24, SSIM=0.7521, LPIPS=0.3287, CLIPIQA=0.2570.
    # Important:
    #   This compares your per-image C=128 results against the reported base mean.
    #   For the strongest proof, also evaluate official LSTMConvSR on this exact
    #   saved split and set BASELINE_PER_IMAGE_CSV to that per-image CSV.
    # ============================
    BASE_UAVID_PSNR_Y: float = 28.24
    BASE_UAVID_SSIM_Y: float = 0.7521
    BASE_UAVID_LPIPS: float = 0.3287
    BASE_UAVID_CLIPIQA: float = 0.2570
    STAT_N_BOOT: int = 10000
    BASELINE_PER_IMAGE_CSV: str = ""
    STAT_FIG_DPI: int = 300

    # Save paths
    WORK_DIR: str = "/kaggle/working/famamba_lstmconvsr_uavid_C128_exact_stat"
    CKPT_NAME: str = "best_famamba_lstmconvsr_x4_uavid_C128_exact_stat.pth"


cfg = CFG()

os.makedirs(cfg.WORK_DIR, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.SEED)

print("========== FINAL FAIR COMPARISON CONFIG ==========")
print("Dataset: UAVid only, exact base-paper split count and identity protocol")
print("SMOKE_TEST:", cfg.SMOKE_TEST)
print("Epochs:", cfg.EPOCHS)
print("Pair HR size:", cfg.HR_CROP)
print("Pair LR size:", cfg.HR_CROP // cfg.SCALE)
print("Train HR crop:", cfg.TRAIN_HR_CROP)
print("Train LR crop:", cfg.TRAIN_HR_CROP // cfg.SCALE)
print("Model width C:", cfg.EMBED_DIM)
print("Effective batch:", cfg.BATCH_SIZE * cfg.ACCUM_STEPS)
print("WORK_DIR:", cfg.WORK_DIR)
print("==================================================")


In [ ]:
# 2. Kaggle input discovery
# ============================================================

from pathlib import Path

# Existing datasets
POTSDAM_ROOT = Path("/kaggle/input/datasets/trito12/potsdam-dataset")
RSSCN7_ROOT  = Path("/kaggle/input/datasets/yangpeng1995/rsscn7")

# UAVid exact-comparison setup:
# Train/Val from patched 512 dataset.
# Test from raw UAVid dataset.
UAVID_PATCHED_ROOT = Path("/kaggle/input/datasets/mastershomya/uavid-patched-512x512")
UAVID_RAW_ROOT     = Path("/kaggle/input/datasets/dasmehdixtr/uavid-v1")

# Keep UAVID_ROOT for compatibility with the existing Section 5 code.
# The updated build_uavid_indices() will internally use both UAVID_PATCHED_ROOT and UAVID_RAW_ROOT.
UAVID_ROOT = UAVID_PATCHED_ROOT

print("Detected Kaggle inputs:")
print("Potsdam          :", POTSDAM_ROOT, "exists:", POTSDAM_ROOT.exists())
print("RSSCN7           :", RSSCN7_ROOT, "exists:", RSSCN7_ROOT.exists())
print("UAVid patched 512:", UAVID_PATCHED_ROOT, "exists:", UAVID_PATCHED_ROOT.exists())
print("UAVid raw        :", UAVID_RAW_ROOT, "exists:", UAVID_RAW_ROOT.exists())

if cfg.USE_POTSDAM and not POTSDAM_ROOT.exists():
    print("[WARN] Potsdam not found. Add trito12/potsdam-dataset from Kaggle Input.")

if cfg.USE_UAVID:
    if not UAVID_PATCHED_ROOT.exists():
        print("[WARN] UAVid patched 512 dataset not found. Add mastershomya/uavid-patched-512x512 from Kaggle Input.")
    if not UAVID_RAW_ROOT.exists():
        print("[WARN] UAVid raw dataset not found. Add dasmehdixtr/uavid-v1 from Kaggle Input.")

if cfg.USE_RSSCN7 and not RSSCN7_ROOT.exists():
    print("[WARN] RSSCN7 not found. Add yangpeng1995/rsscn7 from Kaggle Input.")


# ============================================================
# 3. Utility functions for image loading and crop indexing
# ============================================================

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def list_images(root):
    root = Path(root)
    if root is None or not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS])

def pil_loader_rgb(path):
    img = Image.open(path)
    img = img.convert("RGB")
    return img

def crop_positions(length, crop):
    """
    Non-overlap crops with last crop included.

    For UAVid:
    3840x2160 with 512 crop -> 8 x-positions and 5 y-positions = 40 patches.
    4096x2160 with 512 crop -> 8 x-positions and 5 y-positions = 40 patches.
    """
    if length <= crop:
        return [0]

    positions = list(range(0, length - crop + 1, crop))

    if positions[-1] != length - crop:
        positions.append(length - crop)

    return positions

def make_crop_boxes(width, height, crop):
    xs = crop_positions(width, crop)
    ys = crop_positions(height, crop)

    boxes = []
    for y in ys:
        for x in xs:
            boxes.append((x, y, x + crop, y + crop))

    return boxes

def resize_lr_from_hr(hr_tensor, scale=4):
    # hr_tensor: C,H,W in [0,1]
    c, h, w = hr_tensor.shape

    lr = TF.resize(
        hr_tensor,
        [h // scale, w // scale],
        interpolation=transforms.InterpolationMode.BICUBIC,
        antialias=True
    )

    return lr

def to_tensor(img):
    return TF.to_tensor(img)

def center_crop_or_resize(img, size):
    w, h = img.size

    if w < size or h < size:
        img = TF.resize(
            img,
            [max(size, h), max(size, w)],
            interpolation=transforms.InterpolationMode.BICUBIC
        )
        w, h = img.size

    return TF.center_crop(img, [size, size])

In [ ]:
# 4. Dataset index builders
# ============================================================

def split_list_deterministic(items, seed=42, train_ratio=0.7, val_ratio=0.1):
    rng = random.Random(seed)
    items = list(items)
    rng.shuffle(items)
    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train = items[:n_train]
    val = items[n_train:n_train+n_val]
    test = items[n_train+n_val:]
    return train, val, test


# ============================================================
# RSSCN7
# ============================================================

def build_rsscn7_indices(root, hr_crop):
    images = list_images(root)

    valid = []
    for p in images:
        try:
            img = Image.open(p)
            w, h = img.size
            if min(w, h) >= 100:
                valid.append(p)
        except Exception:
            pass

    valid = sorted(valid)
    rng = random.Random(cfg.SEED)
    rng.shuffle(valid)

    n = len(valid)
    half = n // 2

    train_pool = valid[:half]
    test_files = valid[half:]

    val_n = max(1, int(0.2 * len(train_pool)))
    val_files = train_pool[:val_n]
    train_files = train_pool[val_n:]

    def make_items(files):
        return [
            {
                "dataset": "rsscn7",
                "path": str(p),
                "box": None,
                "hr_crop": hr_crop
            }
            for p in files
        ]

    return make_items(train_files), make_items(val_files), make_items(test_files)


# ============================================================
# Potsdam compatibility section
# Not used in this UAVid run
# ============================================================

POTSDAM_TRAIN_IDS = {"2_11", "3_10", "4_10", "5_10", "6_7", "7_7"}
POTSDAM_VAL_IDS   = {"2_10"}
POTSDAM_TEST_IDS  = {
    "2_13", "2_14", "3_13", "3_14", "4_13", "4_14", "4_15",
    "5_13", "5_14", "5_15", "6_13", "6_14", "6_15", "7_13"
}

def parse_potsdam_tile_id(path):
    import re
    name = Path(path).name.lower()

    m = re.search(r"top_potsdam_(\d+)_(\d+)", name)
    if m:
        return f"{int(m.group(1))}_{int(m.group(2))}"

    m = re.search(r"potsdam_(\d+)_(\d+)", name)
    if m:
        return f"{int(m.group(1))}_{int(m.group(2))}"

    return None


def is_potsdam_rgb(path):
    s = str(path).lower()

    bad = [
        "label", "labels", "mask", "masks",
        "dsm", "ndsm", "nirrg", "irrg",
        "ground_truth", "gt"
    ]

    if any(b in s for b in bad):
        return False

    if Path(path).suffix.lower() not in IMG_EXTS:
        return False

    if "/image/" in s and "top_potsdam_" in s:
        return True

    if "rgb" in s and "top_potsdam_" in s:
        return True

    return False


def build_potsdam_indices(root, hr_crop):
    from collections import Counter

    images = [p for p in list_images(root) if is_potsdam_rgb(p)]

    train, val, test = [], [], []

    id_counter = Counter()
    unknown = 0
    skipped_nonpaper = 0
    skipped_too_small = 0
    open_failed = 0

    for p in images:
        tid = parse_potsdam_tile_id(p)

        if tid is None:
            unknown += 1
            continue

        id_counter[tid] += 1

        try:
            img = Image.open(p)
            w, h = img.size
        except Exception:
            open_failed += 1
            continue

        if tid not in POTSDAM_TRAIN_IDS and tid not in POTSDAM_VAL_IDS and tid not in POTSDAM_TEST_IDS:
            skipped_nonpaper += 1
            continue

        if w < 32 or h < 32:
            skipped_too_small += 1
            continue

        item = {
            "dataset": "potsdam",
            "path": str(p),
            "box": None,
            "hr_crop": hr_crop,
            "tile_id": tid
        }

        if tid in POTSDAM_TRAIN_IDS:
            train.append(item)
        elif tid in POTSDAM_VAL_IDS:
            val.append(item)
        elif tid in POTSDAM_TEST_IDS:
            test.append(item)

    print("Potsdam parser diagnostics:")
    print("  total RGB candidate images:", len(images))
    print("  unknown tile-id images:", unknown)
    print("  open failed:", open_failed)
    print("  skipped non-paper tile images:", skipped_nonpaper)
    print("  skipped too-small images:", skipped_too_small)
    print("  train tile IDs:", sorted(set(x["tile_id"] for x in train)))
    print("  val tile IDs:", sorted(set(x["tile_id"] for x in val)))
    print("  test tile IDs:", sorted(set(x["tile_id"] for x in test)))
    print("  train:", len(train), "val:", len(val), "test:", len(test))

    if len(train) == 0 or len(val) == 0 or len(test) == 0:
        raise RuntimeError("Potsdam split failed. Do not use random split for exact comparison.")

    return train, val, test


# ============================================================
# UAVid exact hybrid builder
# Train/Val: mastershomya/uavid-patched-512x512
# Test: dasmehdixtr/uavid-v1
# Target: train=1600, val=560, test=1200
# ============================================================

UAVID_TRAIN_SEQ = set(list(range(1, 16)) + list(range(31, 36)))
UAVID_VAL_SEQ   = set(list(range(16, 21)) + [36, 37])
UAVID_TEST_SEQ  = set(list(range(21, 31)) + list(range(38, 43)))
UAVID_FRAMES    = {"000000", "000900"}


def parse_uavid_seq_id(path):
    import re
    s = str(path).lower().replace("\\", "/")

    m = re.search(r"(?:seq|sequence)[_\- ]?(\d+)", s)
    if m:
        return int(m.group(1))

    return None


def parse_uavid_frame_id(path):
    import re
    stem = Path(path).stem.lower()

    m = re.search(r"(\d{6})", stem)
    if m:
        return m.group(1)

    digits = "".join([c for c in stem if c.isdigit()])
    if len(digits) >= 6:
        return digits[-6:]

    return None


def parse_patch_xy(path):
    import re
    stem = Path(path).stem.lower()

    m = re.search(r"patch_(\d+)_(\d+)", stem)
    if m:
        return int(m.group(1)), int(m.group(2))

    return None


def is_uavid_image(path):
    s = str(path).lower()

    bad = [
        "label", "labels", "mask", "masks",
        "gt", "annotation", "annotations",
        "class", "classes", "colorlabel"
    ]

    if any(b in s for b in bad):
        return False

    if Path(path).suffix.lower() not in IMG_EXTS:
        return False

    return True


def is_exact_nonoverlap_patch(path):
    xy = parse_patch_xy(path)
    if xy is None:
        return False

    x, y = xy

    allowed_x = {0, 512, 1024, 1536, 2048, 2560, 3072, 3328, 3584}
    allowed_y = {0, 512, 1024, 1536, 1648}

    return x in allowed_x and y in allowed_y


def build_uavid_train_val_from_patched(root, hr_crop):
    from collections import Counter

    images = [p for p in list_images(root) if is_uavid_image(p)]

    train, val = [], []

    seq_counter = Counter()
    frame_counter = Counter()
    kept_counter = Counter()

    skipped_seq = 0
    skipped_frame = 0
    skipped_patch = 0

    for p in images:
        seq_id = parse_uavid_seq_id(p)
        frame_id = parse_uavid_frame_id(p)

        if seq_id is None:
            skipped_seq += 1
            continue

        seq_counter[seq_id] += 1

        if frame_id not in UAVID_FRAMES:
            skipped_frame += 1
            continue

        frame_counter[frame_id] += 1

        if not is_exact_nonoverlap_patch(p):
            skipped_patch += 1
            continue

        item = {
            "dataset": "uavid",
            "path": str(p),
            "box": None,
            "hr_crop": hr_crop,
            "seq_id": seq_id,
            "frame_id": frame_id,
            "source": "patched_train_val"
        }

        if seq_id in UAVID_TRAIN_SEQ:
            train.append(item)
            kept_counter["train"] += 1
        elif seq_id in UAVID_VAL_SEQ:
            val.append(item)
            kept_counter["val"] += 1

    print("UAVid patched train/val diagnostics:")
    print("  root:", root)
    print("  total image candidates:", len(images))
    print("  skipped no-seq:", skipped_seq)
    print("  skipped non-selected frames:", skipped_frame)
    print("  skipped non-exact/overlap patches:", skipped_patch)
    print("  selected frame counts:", dict(frame_counter))
    print("  kept counts:", dict(kept_counter))
    print("  train seq IDs:", sorted(set(x["seq_id"] for x in train)))
    print("  val seq IDs:", sorted(set(x["seq_id"] for x in val)))
    print("  train count:", len(train))
    print("  val count:", len(val))

    return train, val


def build_uavid_test_from_raw(root, hr_crop):
    from collections import Counter

    images = [p for p in list_images(root) if is_uavid_image(p)]

    test = []

    seq_counter = Counter()
    frame_counter = Counter()
    size_counter = Counter()

    skipped_seq = 0
    skipped_frame = 0
    open_failed = 0

    for p in images:
        seq_id = parse_uavid_seq_id(p)
        frame_id = parse_uavid_frame_id(p)

        if seq_id is None:
            skipped_seq += 1
            continue

        seq_counter[seq_id] += 1

        if seq_id not in UAVID_TEST_SEQ:
            continue

        if frame_id not in UAVID_FRAMES:
            skipped_frame += 1
            continue

        frame_counter[frame_id] += 1

        try:
            img = Image.open(p)
            w, h = img.size
            size_counter[(w, h)] += 1
        except Exception:
            open_failed += 1
            continue

        boxes = make_crop_boxes(w, h, hr_crop)

        for box in boxes:
            test.append({
                "dataset": "uavid",
                "path": str(p),
                "box": box,
                "hr_crop": hr_crop,
                "seq_id": seq_id,
                "frame_id": frame_id,
                "source": "raw_test"
            })

    print("UAVid raw test diagnostics:")
    print("  root:", root)
    print("  total image candidates:", len(images))
    print("  skipped no-seq:", skipped_seq)
    print("  skipped non-selected frames in test seqs:", skipped_frame)
    print("  open failed:", open_failed)
    print("  selected frame counts:", dict(frame_counter))
    print("  image sizes:", dict(size_counter))
    print("  test seq IDs:", sorted(set(x["seq_id"] for x in test)))
    print("  test count:", len(test))

    return test


def build_uavid_indices(root, hr_crop):
    train, val = build_uavid_train_val_from_patched(UAVID_PATCHED_ROOT, hr_crop)
    test = build_uavid_test_from_raw(UAVID_RAW_ROOT, hr_crop)

    print("\nFinal UAVid exact-hybrid counts:")
    print("  train:", len(train))
    print("  val  :", len(val))
    print("  test :", len(test))

    print("\nFinal UAVid exact-hybrid sequence check:")
    print("  train seq IDs:", sorted(set(x["seq_id"] for x in train)))
    print("  val seq IDs:", sorted(set(x["seq_id"] for x in val)))
    print("  test seq IDs:", sorted(set(x["seq_id"] for x in test)))
    print("  train frame IDs:", sorted(set(x["frame_id"] for x in train)))
    print("  val frame IDs:", sorted(set(x["frame_id"] for x in val)))
    print("  test frame IDs:", sorted(set(x["frame_id"] for x in test)))

    if len(train) != 1600 or len(val) != 560 or len(test) != 1200:
        print("[WARN] Counts do not exactly match expected 1600/560/1200.")
        print("       Do not start final training until this is fixed.")

    if len(train) == 0 or len(val) == 0 or len(test) == 0:
        raise RuntimeError("UAVid exact-hybrid split failed.")

    return train, val, test

In [ ]:
# ============================================================
# FINAL FIX: Fast UAVid exact-hybrid builder
# Train/Val from patched 512x512 dataset
# Test from raw UAVid dataset
#
# Target:
# train = 1600
# val   = 560
# test  = 1200
# ============================================================

import re
from pathlib import Path
from PIL import Image
from collections import defaultdict

UAVID_TRAIN_SEQ = set(list(range(1, 16)) + list(range(31, 36)))
UAVID_VAL_SEQ   = set(list(range(16, 21)) + [36, 37])
UAVID_TEST_SEQ  = set(list(range(21, 31)) + list(range(38, 43)))
UAVID_FRAMES    = {"000000", "000900"}

def parse_uavid_seq_id(path):
    s = str(path).lower().replace("\\", "/")
    m = re.search(r"(?:seq|sequence)[_\- ]?(\d+)", s)
    if m:
        return int(m.group(1))
    return None

def parse_uavid_frame_id(path):
    stem = Path(path).stem.lower()
    m = re.search(r"(\d{6})", stem)
    if m:
        return m.group(1)
    return None

def parse_patch_yx(path):
    """
    Patched dataset names look like:
    seq18_000400_patch_512_1536.png

    In this Kaggle dataset, the order behaves as:
    patch_y_x
    """
    stem = Path(path).stem.lower()
    m = re.search(r"patch_(\d+)_(\d+)", stem)
    if m:
        y = int(m.group(1))
        x = int(m.group(2))
        return y, x
    return None

def is_uavid_image(path):
    s = str(path).lower()

    bad = [
        "label", "labels", "mask", "masks",
        "gt", "annotation", "annotations",
        "class", "classes", "colorlabel"
    ]

    if any(b in s for b in bad):
        return False

    return Path(path).suffix.lower() in IMG_EXTS

def expected_nonoverlap_positions_for_group(paths):
    """
    For one seq+frame group, select the paper-style 40 non-overlap patches.

    UAVid images are either:
    3840x2160 -> x: 0,512,1024,1536,2048,2560,3072,3328
    4096x2160 -> x: 0,512,1024,1536,2048,2560,3072,3584

    y is:
    0,512,1024,1536,1648

    This function detects whether the group contains x=3584.
    If yes, use 3584 as the last x; otherwise use 3328.
    """
    coords = []
    for p in paths:
        yx = parse_patch_yx(p)
        if yx is not None:
            y, x = yx
            coords.append((y, x))

    xs = {x for y, x in coords}

    if 3584 in xs:
        x_positions = [0, 512, 1024, 1536, 2048, 2560, 3072, 3584]
    else:
        x_positions = [0, 512, 1024, 1536, 2048, 2560, 3072, 3328]

    y_positions = [0, 512, 1024, 1536, 1648]

    return set((y, x) for y in y_positions for x in x_positions)

def build_uavid_train_val_from_patched_fast(root, hr_crop):
    root = Path(root)

    image_dirs = [
        root / "patched_train_dataset_512" / "patched_train_dataset_512" / "images",
        root / "patched_val_dataset_512" / "patched_val_dataset_512" / "images",
    ]

    grouped = defaultdict(list)

    total_files = 0
    skipped_seq = 0
    skipped_frame = 0

    for img_dir in image_dirs:
        if not img_dir.exists():
            print("[WARN] Missing image dir:", img_dir)
            continue

        for p in img_dir.glob("*.png"):
            total_files += 1

            seq_id = parse_uavid_seq_id(p)
            frame_id = parse_uavid_frame_id(p)

            if seq_id is None:
                skipped_seq += 1
                continue

            if frame_id not in UAVID_FRAMES:
                skipped_frame += 1
                continue

            if seq_id not in UAVID_TRAIN_SEQ and seq_id not in UAVID_VAL_SEQ:
                continue

            grouped[(seq_id, frame_id)].append(p)

    train, val = [], []
    bad_groups = []

    for (seq_id, frame_id), paths in sorted(grouped.items()):
        expected_positions = expected_nonoverlap_positions_for_group(paths)

        path_by_yx = {}
        for p in paths:
            yx = parse_patch_yx(p)
            if yx is not None:
                path_by_yx[yx] = p

        selected = []
        missing_positions = []

        for yx in sorted(expected_positions):
            if yx in path_by_yx:
                selected.append(path_by_yx[yx])
            else:
                missing_positions.append(yx)

        if len(selected) != 40:
            bad_groups.append((seq_id, frame_id, len(selected), missing_positions[:10]))

        for p in selected:
            item = {
                "dataset": "uavid",
                "path": str(p),
                "box": None,
                "hr_crop": hr_crop,
                "seq_id": seq_id,
                "frame_id": frame_id,
                "source": "patched_train_val"
            }

            if seq_id in UAVID_TRAIN_SEQ:
                train.append(item)
            elif seq_id in UAVID_VAL_SEQ:
                val.append(item)

    print("Fast UAVid patched train/val diagnostics:")
    print("  total png files scanned:", total_files)
    print("  skipped no-seq:", skipped_seq)
    print("  skipped non-selected frames:", skipped_frame)
    print("  selected seq-frame groups:", len(grouped))
    print("  train count:", len(train))
    print("  val count:", len(val))
    print("  train seq IDs:", sorted(set(x["seq_id"] for x in train)))
    print("  val seq IDs:", sorted(set(x["seq_id"] for x in val)))

    if bad_groups:
        print("[WARN] Some seq-frame groups did not produce 40 patches.")
        print("First 10 bad groups:")
        for item in bad_groups[:10]:
            print(item)

    return train, val

def build_uavid_test_from_raw_fast(root, hr_crop):
    root = Path(root)

    test = []

    image_root = root / "uavid_test"

    if not image_root.exists():
        raise RuntimeError(f"Raw UAVid test folder not found: {image_root}")

    total_frames = 0
    missing_frames = []

    for seq_id in sorted(UAVID_TEST_SEQ):
        seq_dir = image_root / f"seq{seq_id}" / "Images"

        if not seq_dir.exists():
            print("[WARN] Missing raw test sequence dir:", seq_dir)
            continue

        for frame_id in sorted(UAVID_FRAMES):
            p = seq_dir / f"{frame_id}.png"

            if not p.exists():
                missing_frames.append(str(p))
                continue

            total_frames += 1

            img = Image.open(p)
            w, h = img.size

            boxes = make_crop_boxes(w, h, hr_crop)

            for box in boxes:
                test.append({
                    "dataset": "uavid",
                    "path": str(p),
                    "box": box,
                    "hr_crop": hr_crop,
                    "seq_id": seq_id,
                    "frame_id": frame_id,
                    "source": "raw_test"
                })

    print("Fast UAVid raw test diagnostics:")
    print("  raw test frames found:", total_frames)
    print("  missing frames:", len(missing_frames))
    print("  test count:", len(test))
    print("  test seq IDs:", sorted(set(x["seq_id"] for x in test)))
    print("  test frame IDs:", sorted(set(x["frame_id"] for x in test)))

    if missing_frames:
        print("First missing frames:")
        for p in missing_frames[:10]:
            print(p)

    return test

def build_uavid_indices(root, hr_crop):
    train, val = build_uavid_train_val_from_patched_fast(UAVID_PATCHED_ROOT, hr_crop)
    test = build_uavid_test_from_raw_fast(UAVID_RAW_ROOT, hr_crop)

    print("\nFinal UAVid exact-hybrid counts:")
    print("  train:", len(train))
    print("  val  :", len(val))
    print("  test :", len(test))

    if len(train) != 1600 or len(val) != 560 or len(test) != 1200:
        print("[WARN] Counts do not match expected 1600/560/1200.")
        print("       Do not train until fixed.")

    if len(train) == 0 or len(val) == 0 or len(test) == 0:
        raise RuntimeError("UAVid exact-hybrid split failed.")

    return train, val, test

print("[OK] Final UAVid exact-hybrid builder override is active.")

In [ ]:
# 5. Build unified train/val/test item lists
# ============================================================

train_items, val_items, test_items = [], [], []

if cfg.USE_POTSDAM and POTSDAM_ROOT is not None and POTSDAM_ROOT.exists():
    tr, va, te = build_potsdam_indices(POTSDAM_ROOT, cfg.HR_CROP)
    print(f"Potsdam items: train={len(tr)}, val={len(va)}, test={len(te)}")
    train_items += tr
    val_items += va
    test_items += te

if cfg.USE_UAVID and UAVID_ROOT is not None and UAVID_ROOT.exists():
    tr, va, te = build_uavid_indices(UAVID_ROOT, cfg.HR_CROP)
    print(f"UAVid items: train={len(tr)}, val={len(va)}, test={len(te)}")
    train_items += tr
    val_items += va
    test_items += te

if cfg.USE_RSSCN7 and RSSCN7_ROOT is not None and RSSCN7_ROOT.exists():
    tr, va, te = build_rsscn7_indices(RSSCN7_ROOT, cfg.HR_CROP)
    print(f"RSSCN7 items: train={len(tr)}, val={len(va)}, test={len(te)}")
    train_items += tr
    val_items += va
    test_items += te

if len(train_items) == 0:
    raise RuntimeError("No training items found. Add Kaggle datasets through Add Input and rerun.")

# Keep deterministic order after construction.
random.Random(cfg.SEED).shuffle(train_items)
random.Random(cfg.SEED + 1).shuffle(val_items)
random.Random(cfg.SEED + 2).shuffle(test_items)

def limit_items(items, split):
    if cfg.SMOKE_TEST:
        if split == "train":
            return items[:cfg.MAX_TRAIN_SAMPLES_SMOKE]
        if split == "val":
            return items[:cfg.MAX_VAL_SAMPLES_SMOKE]
        if split == "test":
            return items[:cfg.MAX_TEST_SAMPLES_SMOKE]
    else:
        limit = {
            "train": cfg.MAX_TRAIN_SAMPLES_FULL,
            "val": cfg.MAX_VAL_SAMPLES_FULL,
            "test": cfg.MAX_TEST_SAMPLES_FULL
        }[split]
        return items if limit is None else items[:limit]

train_items = limit_items(train_items, "train")
val_items   = limit_items(val_items, "val")
test_items  = limit_items(test_items, "test")

print("Final item counts:")
print("train:", len(train_items))
print("val  :", len(val_items))
print("test :", len(test_items))

# Hard check for exact base-paper UAVid counts.
if cfg.USE_UAVID and not cfg.SMOKE_TEST:
    assert len(train_items) == 1600, f"Expected UAVid train=1600, got {len(train_items)}"
    assert len(val_items) == 560, f"Expected UAVid val=560, got {len(val_items)}"
    assert len(test_items) == 1200, f"Expected UAVid test=1200, got {len(test_items)}"
    print("[OK] UAVid split count exactly matches base paper: 1600 / 560 / 1200")

# Save split identity for reproducibility and paired testing.
def save_split_identity(train_items, val_items, test_items):
    rows = []
    for split_name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
        for idx, item in enumerate(items):
            box = item.get("box", None)
            rows.append({
                "split": split_name,
                "index": idx,
                "dataset": item.get("dataset", ""),
                "source": item.get("source", ""),
                "seq_id": item.get("seq_id", ""),
                "frame_id": item.get("frame_id", ""),
                "path": item.get("path", ""),
                "box_x1": "" if box is None else box[0],
                "box_y1": "" if box is None else box[1],
                "box_x2": "" if box is None else box[2],
                "box_y2": "" if box is None else box[3],
                "hr_crop": item.get("hr_crop", "")
            })
    split_csv = os.path.join(cfg.WORK_DIR, "uavid_exact_split_identity_C128.csv")
    pd.DataFrame(rows).to_csv(split_csv, index=False)
    print("Saved UAVid split identity:", split_csv)

save_split_identity(train_items, val_items, test_items)


In [ ]:
# 6. Dataset class
# ============================================================

class RemoteSensingSRDataset(Dataset):
    def __init__(self, items, scale=4, train=True, augment=True):
        self.items = items
        self.scale = scale
        self.train = train
        self.augment = augment and train

    def __len__(self):
        return len(self.items)

    def _load_full_hr(self, item):
        img = pil_loader_rgb(item["path"])
        hr_size = item["hr_crop"]

        if item["box"] is not None:
            box = item["box"]
            img = img.crop(box)
        else:
            img = center_crop_or_resize(img, hr_size)

        if img.size != (hr_size, hr_size):
            img = TF.resize(
                img,
                [hr_size, hr_size],
                interpolation=transforms.InterpolationMode.BICUBIC
            )

        return img

    def _paired_random_crop(self, lr, hr, train_hr_crop, scale):
        """
        lr: C,H,W
        hr: C,scale*H,scale*W
        train_hr_crop: e.g., 128
        train_lr_crop: e.g., 32
        """
        train_lr_crop = train_hr_crop // scale

        _, lr_h, lr_w = lr.shape

        if lr_h < train_lr_crop or lr_w < train_lr_crop:
            return lr, hr

        top = random.randint(0, lr_h - train_lr_crop)
        left = random.randint(0, lr_w - train_lr_crop)

        lr_crop = lr[:, top:top + train_lr_crop, left:left + train_lr_crop]

        hr_top = top * scale
        hr_left = left * scale
        hr_crop = hr[
            :,
            hr_top:hr_top + train_hr_crop,
            hr_left:hr_left + train_hr_crop
        ]

        return lr_crop, hr_crop

    def _make_item_id(self, item):
        seq = item.get("seq_id", "NA")
        frame = item.get("frame_id", "NA")
        source = item.get("source", "NA")
        box = item.get("box", None)
        if box is None:
            box_txt = "patch"
        else:
            box_txt = f"x{box[0]}_y{box[1]}_x{box[2]}_y{box[3]}"
        stem = Path(item["path"]).stem
        return f"{item.get('dataset','data')}_{source}_seq{seq}_{frame}_{box_txt}_{stem}"

    def __getitem__(self, idx):
        item = self.items[idx]

        # Full HR pair image:
        # UAVid: 512x512
        hr_img = self._load_full_hr(item)

        if self.augment:
            if random.random() < 0.5:
                hr_img = TF.hflip(hr_img)
            if random.random() < 0.5:
                hr_img = TF.vflip(hr_img)
            if random.random() < 0.5:
                angle = random.choice([0, 90, 180, 270])
                hr_img = TF.rotate(hr_img, angle)

        hr = to_tensor(hr_img)

        # Full LR pair:
        # UAVid: 128x128
        lr = resize_lr_from_hr(hr, scale=self.scale)

        # Training crop:
        # LR 32x32, HR 128x128
        if self.train:
            train_hr_crop = getattr(cfg, "TRAIN_HR_CROP", 128)
            lr, hr = self._paired_random_crop(
                lr,
                hr,
                train_hr_crop=train_hr_crop,
                scale=self.scale
            )

        return lr, hr, self._make_item_id(item)

train_ds = RemoteSensingSRDataset(train_items, scale=cfg.SCALE, train=True, augment=True)
val_ds   = RemoteSensingSRDataset(val_items, scale=cfg.SCALE, train=False, augment=False)
test_ds  = RemoteSensingSRDataset(test_items, scale=cfg.SCALE, train=False, augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True
)


In [ ]:
# 7. Model blocks
# ============================================================

class LayerNorm2d(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(dim, eps=eps)

    def forward(self, x):
        # B,C,H,W -> B,H,W,C -> B,C,H,W
        return self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2).contiguous()

class ResidualCNNBlock(nn.Module):
    def __init__(self, dim, res_scale=1.0):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(1) * res_scale)
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1),
            nn.PReLU(num_parameters=1, init=0.2),
            nn.Conv2d(dim, dim, 3, 1, 1),
        )

    def forward(self, x):
        return x + self.scale * self.net(x)

class LocalCNNBranch(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(1) * 0.1)
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 1, 1, 0),
            nn.GELU(),
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim),
            nn.GELU(),
            nn.Conv2d(dim, dim, 1, 1, 0),
        )

    def forward(self, x):
        return x + self.scale * self.net(x)

# ----------------------------
# Haar DWT / IDWT
# ----------------------------
def pad_to_even(x):
    b, c, h, w = x.shape
    pad_h = h % 2
    pad_w = w % 2
    if pad_h or pad_w:
        x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
    return x, (h, w)

def haar_dwt2(x):
    x, orig_hw = pad_to_even(x)
    x00 = x[:, :, 0::2, 0::2]
    x01 = x[:, :, 0::2, 1::2]
    x10 = x[:, :, 1::2, 0::2]
    x11 = x[:, :, 1::2, 1::2]

    ll = (x00 + x01 + x10 + x11) / 2.0
    lh = (x00 - x01 + x10 - x11) / 2.0
    hl = (x00 + x01 - x10 - x11) / 2.0
    hh = (x00 - x01 - x10 + x11) / 2.0
    return ll, lh, hl, hh, orig_hw

def haar_idwt2(ll, lh, hl, hh, orig_hw=None):
    x00 = (ll + lh + hl + hh) / 2.0
    x01 = (ll - lh + hl - hh) / 2.0
    x10 = (ll + lh - hl - hh) / 2.0
    x11 = (ll - lh - hl + hh) / 2.0

    b, c, h, w = ll.shape
    out = torch.zeros(b, c, h * 2, w * 2, device=ll.device, dtype=ll.dtype)
    out[:, :, 0::2, 0::2] = x00
    out[:, :, 0::2, 1::2] = x01
    out[:, :, 1::2, 0::2] = x10
    out[:, :, 1::2, 1::2] = x11

    if orig_hw is not None:
        oh, ow = orig_hw
        out = out[:, :, :oh, :ow]
    return out

class FrequencyBranch(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.att = nn.Sequential(
            nn.Conv2d(dim * 3, dim * 3, 3, 1, 1, groups=3),
            nn.GELU(),
            nn.Conv2d(dim * 3, dim * 3, 1, 1, 0),
            nn.Sigmoid()
        )
        self.proj = nn.Conv2d(dim, dim, 1, 1, 0)
        self.scale = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x):
        ll, lh, hl, hh, orig_hw = haar_dwt2(x)
        hf = torch.cat([lh, hl, hh], dim=1)
        a = self.att(hf)
        lh2, hl2, hh2 = torch.chunk(hf * a, 3, dim=1)

        zero_ll = torch.zeros_like(ll)
        detail = haar_idwt2(zero_ll, lh2, hl2, hh2, orig_hw=orig_hw)
        detail = self.proj(detail)
        return x + self.scale * detail

# ----------------------------
# Mamba-lite selective scan
# Pure PyTorch implementation.
# Directional row/column scans.
# ----------------------------
# ============================================================
# Fast vectorized Mamba-style 2D branch
# No Python recurrent loop, much faster on Kaggle T4.
# This is a practical fast Mamba-like gated long-range branch.
# ============================================================

class MambaLite2D(nn.Module):
    def __init__(self, dim, kernel_size=9):
        super().__init__()

        self.norm = LayerNorm2d(dim)

        self.in_proj = nn.Conv2d(dim, dim * 2, 1, 1, 0)

        self.dwconv2d = nn.Conv2d(
            dim, dim, 3, 1, 1,
            groups=dim
        )

        # Horizontal sequence mixing
        self.row_mixer = nn.Conv1d(
            dim, dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=dim
        )

        # Vertical sequence mixing
        self.col_mixer = nn.Conv1d(
            dim, dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=dim
        )

        self.channel_mixer = nn.Sequential(
            nn.Conv2d(dim, dim * 2, 1, 1, 0),
            nn.GELU(),
            nn.Conv2d(dim * 2, dim, 1, 1, 0)
        )

        self.out_proj = nn.Conv2d(dim, dim, 1, 1, 0)

        self.scale = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x):
        identity = x

        x = self.norm(x)
        u, z = self.in_proj(x).chunk(2, dim=1)

        u = self.dwconv2d(u)
        z = torch.sigmoid(z)

        b, c, h, w = u.shape

        # Row-wise long-range mixing
        row = u.permute(0, 2, 1, 3).reshape(b * h, c, w)
        row = self.row_mixer(row)
        row = row.reshape(b, h, c, w).permute(0, 2, 1, 3).contiguous()

        # Column-wise long-range mixing
        col = u.permute(0, 3, 1, 2).reshape(b * w, c, h)
        col = self.col_mixer(col)
        col = col.reshape(b, w, c, h).permute(0, 2, 3, 1).contiguous()

        y = (row + col) * 0.5

        # Mamba-style gated output
        y = y * z
        y = self.channel_mixer(y)
        y = self.out_proj(y)

        return identity + self.scale * y

# ----------------------------
# ============================================================
# Fast Vision-LSTM-style guidance branch
# Uses native nn.LSTM on T4 GPU
# ============================================================

class VisionLSTMGuidance(nn.Module):
    def __init__(self, dim, mlp_ratio=2.0):
        super().__init__()

        self.hwd_conv = nn.Sequential(
            nn.Conv2d(dim * 4, dim, 1, 1, 0),
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)
        )

        self.ln = nn.LayerNorm(dim)

        self.lstm = nn.LSTM(
            input_size=dim,
            hidden_size=dim // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.att_conv = nn.Conv2d(dim, 1, 3, 1, 1)

        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Conv2d(dim, hidden, 1, 1, 0),
            nn.GELU(),
            nn.Conv2d(hidden, dim, 1, 1, 0)
        )

        self.alpha1 = nn.Parameter(torch.ones(1) * 0.1)
        self.alpha2 = nn.Parameter(torch.ones(1) * 0.1)

    def hwd(self, x):
        ll, lh, hl, hh, orig_hw = haar_dwt2(x)
        cat = torch.cat([ll, lh, hl, hh], dim=1)
        return self.hwd_conv(cat), orig_hw

    def forward(self, x):
        b, c, h, w = x.shape

        x_low, orig_hw = self.hwd(x)
        _, _, h2, w2 = x_low.shape

        tokens = x_low.flatten(2).transpose(1, 2)
        tokens = self.ln(tokens)

        tokens, _ = self.lstm(tokens)

        y = tokens.transpose(1, 2).reshape(b, c, h2, w2)

        y_up = F.interpolate(y, size=(h, w), mode="bilinear", align_corners=False)
        att = torch.sigmoid(self.att_conv(y_up))

        guided = x * att

        out = x + self.alpha1 * guided
        out = out + self.alpha2 * self.mlp(out)

        return out

# ----------------------------
# FAMCB and FAMCRG
# ----------------------------
class FAMCB(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.cnn = LocalCNNBranch(dim)
        self.mamba = MambaLite2D(dim)
        self.freq = FrequencyBranch(dim)

        # 4 branch gates: identity, cnn, mamba, freq
        self.gate = nn.Conv2d(dim * 4, 4, 1, 1, 0)
        self.fuse = nn.Conv2d(dim * 4, dim, 3, 1, 1)
        self.scale = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x):
        x_cnn = self.cnn(x)
        x_mam = self.mamba(x)
        x_frq = self.freq(x)

        cat = torch.cat([x, x_cnn, x_mam, x_frq], dim=1)
        gates = self.gate(cat)
        gates = F.softmax(gates, dim=1)

        g0 = gates[:, 0:1]
        g1 = gates[:, 1:2]
        g2 = gates[:, 2:3]
        g3 = gates[:, 3:4]

        fused = torch.cat([
            g0 * x,
            g1 * x_cnn,
            g2 * x_mam,
            g3 * x_frq
        ], dim=1)

        y = self.fuse(fused)
        return x + self.scale * y

class FAMCRG(nn.Module):
    def __init__(self, dim, num_blocks=4):
        super().__init__()
        self.blocks = nn.Sequential(*[FAMCB(dim) for _ in range(num_blocks)])
        self.conv = nn.Conv2d(dim, dim, 3, 1, 1)

    def forward(self, x):
        return x + self.conv(self.blocks(x))

class GatedTopDownFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Conv2d(dim * 2, dim, 1, 1, 0)
        self.fuse = nn.Conv2d(dim * 2, dim, 3, 1, 1)
        self.scale = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x_in, x_vilg, x_refined):
        g = torch.sigmoid(self.gate(torch.cat([x_vilg, x_refined], dim=1)))
        y = self.fuse(torch.cat([g * x_vilg, (1.0 - g) * x_refined], dim=1))
        return x_in + self.scale * y

class HybridNeuralBlock(nn.Module):
    def __init__(self, dim, famcb_num=4, mlp_ratio=2.0):
        super().__init__()
        self.vilg = VisionLSTMGuidance(dim, mlp_ratio=mlp_ratio)
        self.famcrg = FAMCRG(dim, num_blocks=famcb_num)
        self.fusion = GatedTopDownFusion(dim)

    def forward(self, x):
        y = self.vilg(x)
        r = self.famcrg(y)
        out = self.fusion(x, y, r)
        return out

class Upsample(nn.Sequential):
    def __init__(self, scale, num_feat):
        m = []
        if scale == 2:
            m += [nn.Conv2d(num_feat, 4 * num_feat, 3, 1, 1), nn.PixelShuffle(2)]
        elif scale == 3:
            m += [nn.Conv2d(num_feat, 9 * num_feat, 3, 1, 1), nn.PixelShuffle(3)]
        elif scale == 4:
            m += [
                nn.Conv2d(num_feat, 4 * num_feat, 3, 1, 1), nn.PixelShuffle(2),
                nn.Conv2d(num_feat, 4 * num_feat, 3, 1, 1), nn.PixelShuffle(2),
            ]
        else:
            raise ValueError("Only scale 2, 3, 4 supported.")
        super().__init__(*m)

class FAMambaLSTMConvSR(nn.Module):
    def __init__(
        self,
        upscale=4,
        in_chans=3,
        embed_dim=64,
        num_blocks=4,
        famcb_per_group=4,
        mlp_ratio=2.0,
        img_range=1.0,
        rgb_mean=(0.4488, 0.4371, 0.4040)
    ):
        super().__init__()
        self.upscale = upscale
        self.img_range = img_range
        self.register_buffer("mean", torch.tensor(rgb_mean).view(1, 3, 1, 1), persistent=False)

        self.conv_first = nn.Conv2d(in_chans, embed_dim, 3, 1, 1)

        self.blocks = nn.ModuleList([
            HybridNeuralBlock(embed_dim, famcb_num=famcb_per_group, mlp_ratio=mlp_ratio)
            for _ in range(num_blocks)
        ])

        self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, 3, 1, 1)

        num_feat = 64
        self.conv_before_upsample = nn.Sequential(
            nn.Conv2d(embed_dim, num_feat, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.upsample = Upsample(upscale, num_feat)
        self.conv_last = nn.Conv2d(num_feat, in_chans, 3, 1, 1)

    def forward_features(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x

    def forward(self, x):
        mean = self.mean.type_as(x)
        x = (x - mean) * self.img_range

        x_first = self.conv_first(x)
        body = self.conv_after_body(self.forward_features(x_first))
        x = x_first + body

        x = self.conv_before_upsample(x)
        x = self.upsample(x)
        x = self.conv_last(x)

        x = x / self.img_range + mean
        return x

In [ ]:
# 8. Loss functions
# ============================================================

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target) ** 2 + self.eps ** 2))

def frequency_loss(pred, target):
    _, p_lh, p_hl, p_hh, _ = haar_dwt2(pred)
    _, t_lh, t_hl, t_hh, _ = haar_dwt2(target)
    return (F.l1_loss(p_lh, t_lh) + F.l1_loss(p_hl, t_hl) + F.l1_loss(p_hh, t_hh)) / 3.0

def sobel_kernels(device, dtype):
    sx = torch.tensor([[-1, 0, 1],
                       [-2, 0, 2],
                       [-1, 0, 1]], device=device, dtype=dtype).view(1, 1, 3, 3)
    sy = torch.tensor([[-1, -2, -1],
                       [0, 0, 0],
                       [1, 2, 1]], device=device, dtype=dtype).view(1, 1, 3, 3)
    return sx, sy

def gradient_map(x):
    # grayscale for edge loss
    gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
    sx, sy = sobel_kernels(x.device, x.dtype)
    gx = F.conv2d(gray, sx, padding=1)
    gy = F.conv2d(gray, sy, padding=1)
    return gx, gy

def edge_loss(pred, target):
    pgx, pgy = gradient_map(pred)
    tgx, tgy = gradient_map(target)
    return F.l1_loss(pgx, tgx) + F.l1_loss(pgy, tgy)


In [ ]:
# 9. Metrics
# ============================================================

def rgb_to_y(x):
    # x: B,3,H,W in [0,1]
    y = 0.257 * x[:, 0:1] + 0.504 * x[:, 1:2] + 0.098 * x[:, 2:3] + 16.0 / 255.0
    return y

def calc_psnr(pred, target, eps=1e-10):
    pred = torch.clamp(pred, 0, 1)
    target = torch.clamp(target, 0, 1)
    mse = F.mse_loss(pred, target)
    return 10.0 * torch.log10(1.0 / (mse + eps))

def gaussian_window(window_size=11, sigma=1.5, channels=1, device="cpu", dtype=torch.float32):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window = g[:, None] @ g[None, :]
    window = window.view(1, 1, window_size, window_size)
    window = window.repeat(channels, 1, 1, 1)
    return window

def calc_ssim(pred, target, window_size=11):
    pred = torch.clamp(pred, 0, 1)
    target = torch.clamp(target, 0, 1)
    b, c, h, w = pred.shape
    window = gaussian_window(window_size, 1.5, c, pred.device, pred.dtype)
    mu1 = F.conv2d(pred, window, padding=window_size//2, groups=c)
    mu2 = F.conv2d(target, window, padding=window_size//2, groups=c)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu12 = mu1 * mu2

    sigma1_sq = F.conv2d(pred * pred, window, padding=window_size//2, groups=c) - mu1_sq
    sigma2_sq = F.conv2d(target * target, window, padding=window_size//2, groups=c) - mu2_sq
    sigma12 = F.conv2d(pred * target, window, padding=window_size//2, groups=c) - mu12

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    ssim_map = ((2 * mu12 + c1) * (2 * sigma12 + c2)) / ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2))
    return ssim_map.mean()

def calc_ms_ssim(pred, target, levels=4):
    weights = torch.tensor([0.0448, 0.2856, 0.3001, 0.2363], device=pred.device, dtype=pred.dtype)
    mssim = []
    x, y = pred, target
    for _ in range(levels):
        mssim.append(torch.clamp(calc_ssim(x, y), 1e-6, 1.0))
        if min(x.shape[-2:]) <= 16:
            break
        x = F.avg_pool2d(x, kernel_size=2, stride=2)
        y = F.avg_pool2d(y, kernel_size=2, stride=2)
    weights = weights[:len(mssim)]
    weights = weights / weights.sum()
    vals = torch.stack(mssim)
    return torch.prod(vals ** weights)

def calc_ergas(pred, target, scale=4, eps=1e-8):
    pred = torch.clamp(pred, 0, 1)
    target = torch.clamp(target, 0, 1)
    # B,C,H,W
    rmse = torch.sqrt(torch.mean((pred - target) ** 2, dim=(0, 2, 3)) + eps)
    mean = torch.mean(target, dim=(0, 2, 3)) + eps
    ergas = 100.0 / scale * torch.sqrt(torch.mean((rmse / mean) ** 2))
    return ergas

def calc_gmsd(pred, target, c=0.0026):
    pred = torch.clamp(pred, 0, 1)
    target = torch.clamp(target, 0, 1)

    pgx, pgy = gradient_map(pred)
    tgx, tgy = gradient_map(target)

    pmag = torch.sqrt(pgx ** 2 + pgy ** 2 + 1e-12)
    tmag = torch.sqrt(tgx ** 2 + tgy ** 2 + 1e-12)

    gms = (2 * pmag * tmag + c) / (pmag ** 2 + tmag ** 2 + c)
    return torch.std(gms)

# Optional pyiqa metrics
class OptionalIqaMetrics:
    def __init__(self, device):
        self.device = device
        self.lpips = None
        self.dists = None
        self.clipiqa = None

        if PYIQA_AVAILABLE:
            for name in ["lpips", "dists", "clipiqa"]:
                try:
                    metric = pyiqa.create_metric(name, device=device)
                    setattr(self, name, metric)
                    print(f"[OK] pyiqa metric loaded: {name}")
                except Exception as e:
                    print(f"[WARN] Could not load pyiqa metric {name}: {e}")
        else:
            print("[WARN] pyiqa unavailable. LPIPS/DISTS/CLIPIQA will be skipped.")

    @torch.no_grad()
    def compute(self, pred, target):
        out = {}
        # pyiqa expects RGB [0,1], B,C,H,W
        if self.lpips is not None:
            try:
                out["lpips"] = float(self.lpips(pred, target).mean().item())
            except Exception:
                out["lpips"] = None
        else:
            out["lpips"] = None

        if self.dists is not None:
            try:
                out["dists"] = float(self.dists(pred, target).mean().item())
            except Exception:
                out["dists"] = None
        else:
            out["dists"] = None

        if self.clipiqa is not None:
            try:
                out["clipiqa"] = float(self.clipiqa(pred).mean().item())
            except Exception:
                out["clipiqa"] = None
        else:
            out["clipiqa"] = None

        return out

iqa_metrics = OptionalIqaMetrics(DEVICE)

In [ ]:
# 10. Build model, optimizer, scheduler
# ============================================================

model = FAMambaLSTMConvSR(
    upscale=cfg.SCALE,
    in_chans=3,
    embed_dim=cfg.EMBED_DIM,
    num_blocks=cfg.NUM_BLOCKS,
    famcb_per_group=cfg.FAMCB_PER_GROUP,
    mlp_ratio=cfg.MLP_RATIO
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params / 1e6:.3f} M")

criterion_rec = CharbonnierLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.LR,
    betas=(0.9, 0.99),
    weight_decay=cfg.WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=250,
    gamma=0.5
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# FLOPs
# FLOPs
# Base paper reports FLOPs using LR input size 1 x 3 x 128 x 128.
# So we also use 128x128 LR dummy input for fair complexity comparison.
if THOP_AVAILABLE:
    try:
        dummy = torch.randn(1, 3, 128, 128).to(DEVICE)
        macs, params = thop.profile(model, inputs=(dummy,), verbose=False)
        print(f"THOP MACs: {macs / 1e9:.3f} G, Params: {params / 1e6:.3f} M")
    except Exception as e:
        print("[WARN] THOP FLOPs failed:", e)

In [ ]:
print("UAVid train:", len(train_items))
print("UAVid val:", len(val_items))
print("UAVid test:", len(test_items))

for split_name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
    print("\n", split_name)
    print("seq IDs:", sorted(set(x.get("seq_id", "NA") for x in items)))
    print("frame IDs:", sorted(set(x.get("frame_id", "NA") for x in items)))
    print("sources:", sorted(set(x.get("source", "NA") for x in items)))
    for x in items[:5]:
        print(x["path"], "seq", x.get("seq_id", "NA"), "frame", x.get("frame_id", "NA"), "box", x.get("box"))


In [ ]:
# 11. Train and evaluation loops
# ============================================================

def compute_total_loss(sr, hr):
    loss_rec = criterion_rec(sr, hr)
    loss_freq = frequency_loss(sr, hr)
    loss_edge = edge_loss(sr, hr)
    loss = loss_rec + cfg.LAMBDA_FREQ * loss_freq + cfg.LAMBDA_EDGE * loss_edge
    return loss, {
        "rec": loss_rec.item(),
        "freq": loss_freq.item(),
        "edge": loss_edge.item()
    }

@torch.no_grad()
def evaluate(model, loader, split_name="val", max_batches=None, use_iqa=True):
    model.eval()
    sums = {
        "psnr_y": 0.0,
        "ssim_y": 0.0,
        "ms_ssim_rgb": 0.0,
        "ergas": 0.0,
        "gmsd": 0.0,
        "lpips": 0.0,
        "dists": 0.0,
        "clipiqa": 0.0,
    }
    counts = {k: 0 for k in sums.keys()}

    start = time.time()
    img_count = 0

    for bi, (lr, hr, names) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break

        lr = lr.to(DEVICE, non_blocking=True)
        hr = hr.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            sr = model(lr)

        sr = torch.clamp(sr, 0, 1)
        hr = torch.clamp(hr, 0, 1)

        # Match dimensions if odd rounding occurs
        if sr.shape[-2:] != hr.shape[-2:]:
            sr = F.interpolate(sr, size=hr.shape[-2:], mode="bicubic", align_corners=False)

        y_sr = rgb_to_y(sr)
        y_hr = rgb_to_y(hr)

        vals = {
            "psnr_y": float(calc_psnr(y_sr, y_hr).item()),
            "ssim_y": float(calc_ssim(y_sr, y_hr).item()),
            "ms_ssim_rgb": float(calc_ms_ssim(sr, hr).item()),
            "ergas": float(calc_ergas(sr, hr, scale=cfg.SCALE).item()),
            "gmsd": float(calc_gmsd(sr, hr).item()),
        }

        for k, v in vals.items():
            sums[k] += v
            counts[k] += 1

        if use_iqa:
            opt = iqa_metrics.compute(sr, hr)
            for k in ["lpips", "dists", "clipiqa"]:
                if opt[k] is not None:
                    sums[k] += opt[k]
                    counts[k] += 1

        img_count += lr.size(0)

    elapsed = time.time() - start
    result = {}
    for k in sums:
        result[k] = sums[k] / max(1, counts[k]) if counts[k] > 0 else None

    result["fps"] = img_count / max(1e-6, elapsed)
    return result

def train_one_epoch(model, loader, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    rec_sum = 0.0
    freq_sum = 0.0
    edge_sum = 0.0
    n = 0

    start = time.time()

    for step, (lr, hr, names) in enumerate(loader):
        lr = lr.to(DEVICE, non_blocking=True)
        hr = hr.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            sr = model(lr)
            if sr.shape[-2:] != hr.shape[-2:]:
                sr = F.interpolate(sr, size=hr.shape[-2:], mode="bicubic", align_corners=False)
            loss, loss_dict = compute_total_loss(sr, hr)
            loss = loss / cfg.ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % cfg.ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        loss_sum += float(loss.item()) * cfg.ACCUM_STEPS
        rec_sum += loss_dict["rec"]
        freq_sum += loss_dict["freq"]
        edge_sum += loss_dict["edge"]
        n += 1

        if step % 50 == 0:
            print(
                f"Epoch {epoch} Step {step}/{len(loader)} "
                f"loss={loss_sum/max(1,n):.5f} "
                f"rec={rec_sum/max(1,n):.5f} "
                f"freq={freq_sum/max(1,n):.5f} "
                f"edge={edge_sum/max(1,n):.5f}"
            )

    # last incomplete accumulation
    if len(loader) % cfg.ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    elapsed = time.time() - start
    return {
        "loss": loss_sum / max(1, n),
        "rec": rec_sum / max(1, n),
        "freq": freq_sum / max(1, n),
        "edge": edge_sum / max(1, n),
        "time": elapsed
    }

@torch.no_grad()
def evaluate_per_image(model, loader, split_name="test", max_batches=None, use_iqa=True):
    """
    Returns aggregate metrics and a per-image DataFrame for statistical analysis.
    """
    model.eval()
    rows = []

    start = time.time()
    img_count = 0

    for bi, (lr, hr, names) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break

        lr = lr.to(DEVICE, non_blocking=True)
        hr = hr.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            sr = model(lr)

        sr = torch.clamp(sr, 0, 1)
        hr = torch.clamp(hr, 0, 1)

        if sr.shape[-2:] != hr.shape[-2:]:
            sr = F.interpolate(sr, size=hr.shape[-2:], mode="bicubic", align_corners=False)

        y_sr = rgb_to_y(sr)
        y_hr = rgb_to_y(hr)

        row = {
            "split": split_name,
            "name": names[0] if isinstance(names, (list, tuple)) else str(names),
            "psnr_y": float(calc_psnr(y_sr, y_hr).item()),
            "ssim_y": float(calc_ssim(y_sr, y_hr).item()),
            "ms_ssim_rgb": float(calc_ms_ssim(sr, hr).item()),
            "ergas": float(calc_ergas(sr, hr, scale=cfg.SCALE).item()),
            "gmsd": float(calc_gmsd(sr, hr).item()),
        }

        if use_iqa:
            opt = iqa_metrics.compute(sr, hr)
            row.update({
                "lpips": opt.get("lpips", None),
                "dists": opt.get("dists", None),
                "clipiqa": opt.get("clipiqa", None),
            })
        else:
            row.update({"lpips": None, "dists": None, "clipiqa": None})

        rows.append(row)
        img_count += lr.size(0)

    elapsed = time.time() - start
    df = pd.DataFrame(rows)

    result = {}
    for col in ["psnr_y", "ssim_y", "ms_ssim_rgb", "ergas", "gmsd", "lpips", "dists", "clipiqa"]:
        if col in df.columns and df[col].notna().any():
            result[col] = float(df[col].dropna().mean())
        else:
            result[col] = None

    result["fps"] = img_count / max(1e-6, elapsed)
    return result, df


In [ ]:
best_psnr = -1.0
best_path = os.path.join(cfg.WORK_DIR, cfg.CKPT_NAME)

VAL_EVERY = 10

print("Starting training...")
print("SMOKE_TEST:", cfg.SMOKE_TEST)
print("Epochs:", cfg.EPOCHS)
print("HR crop:", cfg.HR_CROP, "LR crop:", cfg.HR_CROP // cfg.SCALE)

for epoch in range(1, cfg.EPOCHS + 1):
    train_stats = train_one_epoch(model, train_loader, epoch)
    scheduler.step()

    should_validate = (
        epoch == 1 or
        epoch % VAL_EVERY == 0 or
        epoch == cfg.EPOCHS
    )

    if should_validate:
        val_stats = evaluate(
            model,
            val_loader,
            split_name="val",
            max_batches=cfg.MAX_VAL_SAMPLES_SMOKE if cfg.SMOKE_TEST else None,
            use_iqa=False
        )

        print("=" * 80)
        print(f"Epoch {epoch}/{cfg.EPOCHS}")
        print(f"Train: {train_stats}")
        print(f"Val:   {val_stats}")
        print(f"LR:    {optimizer.param_groups[0]['lr']:.8f}")
        print("=" * 80)

        psnr = val_stats["psnr_y"]

        if psnr is not None and psnr > best_psnr:
            best_psnr = psnr
            torch.save({
                "model": model.state_dict(),
                "cfg": cfg.__dict__,
                "best_psnr": best_psnr,
                "epoch": epoch
            }, best_path)

            print(f"[OK] Saved best checkpoint: {best_path}, PSNR_Y={best_psnr:.4f}")

    else:
        print(
            f"Epoch {epoch}/{cfg.EPOCHS} | "
            f"Train loss={train_stats['loss']:.5f} | "
            f"rec={train_stats['rec']:.5f} | "
            f"freq={train_stats['freq']:.5f} | "
            f"edge={train_stats['edge']:.5f} | "
            f"LR={optimizer.param_groups[0]['lr']:.8f}"
        )

print("Training finished.")
print("Best validation PSNR_Y:", best_psnr)
print("Checkpoint:", best_path)

In [ ]:
# 13. Final test evaluation + per-image metrics for statistics
# ============================================================

if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded best checkpoint from epoch {ckpt.get('epoch')} with PSNR {ckpt.get('best_psnr')}")

test_stats, per_image_df = evaluate_per_image(
    model,
    test_loader,
    split_name="test",
    max_batches=cfg.MAX_TEST_SAMPLES_SMOKE if cfg.SMOKE_TEST else None,
    use_iqa=PYIQA_AVAILABLE
)

per_image_csv = os.path.join(cfg.WORK_DIR, "per_image_test_metrics_UAVid_C128.csv")
per_image_df.to_csv(per_image_csv, index=False)

print("=" * 80)
print("FINAL TEST RESULTS")
for k, v in test_stats.items():
    print(f"{k}: {v}")
print("=" * 80)
print("Saved per-image metrics:", per_image_csv)
display(per_image_df.head())


In [ ]:
import pandas as pd

result_csv = os.path.join(cfg.WORK_DIR, "final_test_results_UAVid_C128.csv")

row = {
    "model": "FAMamba-LSTMConvSR-C128",
    "dataset": "UAVid",
    "epochs": cfg.EPOCHS,
    "hr_crop": cfg.HR_CROP,
    "train_hr_crop": cfg.TRAIN_HR_CROP,
    "embed_dim": cfg.EMBED_DIM,
    "famcb_per_group": cfg.FAMCB_PER_GROUP,
    "batch_size": cfg.BATCH_SIZE,
    "accum_steps": cfg.ACCUM_STEPS,
    "effective_batch": cfg.BATCH_SIZE * cfg.ACCUM_STEPS,
    "use_potsdam": cfg.USE_POTSDAM,
    "use_uavid": cfg.USE_UAVID,
    "use_rsscn7": cfg.USE_RSSCN7,
    "num_test_images": len(per_image_df),
    "per_image_metrics_csv": per_image_csv,
}

row.update(test_stats)

df = pd.DataFrame([row])
df.to_csv(result_csv, index=False)

print("Saved final result CSV:", result_csv)
display(df)


In [ ]:
# ============================================================
# 13B. Strong statistical analysis + publication-ready figures
# ============================================================
# Goal:
# 1) Always perform one-sample statistical testing against the reported
#    LSTMConvSR UAVid mean values from the base paper.
# 2) If you later provide a per-image CSV from the base model on the SAME split,
#    this cell automatically performs stronger paired analysis.
#
# Important for manuscript honesty:
# - One-sample analysis vs reported means is useful evidence, but it is NOT as strong
#   as paired image-wise testing against base-model outputs on the same split.
# - For the strongest MDPI/Q1-style claim, run official LSTMConvSR on this exact
#   saved split and put its per-image CSV path in cfg.BASELINE_PER_IMAGE_CSV.

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
    SCIPY_STATS_AVAILABLE = True
except Exception as e:
    SCIPY_STATS_AVAILABLE = False
    print("[WARN] scipy.stats unavailable:", e)

STAT_DIR = os.path.join(cfg.WORK_DIR, "statistical_analysis_figures")
os.makedirs(STAT_DIR, exist_ok=True)

BASELINE = {
    "psnr_y":  {"value": cfg.BASE_UAVID_PSNR_Y,  "direction": "higher", "label": "PSNR-Y (dB)"},
    "ssim_y":  {"value": cfg.BASE_UAVID_SSIM_Y,  "direction": "higher", "label": "SSIM-Y"},
    "lpips":   {"value": cfg.BASE_UAVID_LPIPS,   "direction": "lower",  "label": "LPIPS"},
    "clipiqa": {"value": cfg.BASE_UAVID_CLIPIQA, "direction": "higher", "label": "CLIPIQA"},
}

def _finite_array(x):
    arr = np.asarray(x, dtype=np.float64)
    return arr[np.isfinite(arr)]

def _positive_improvement(values, baseline_values, direction):
    values = _finite_array(values)
    baseline_values = np.asarray(baseline_values, dtype=np.float64)
    if baseline_values.ndim == 0:
        base = baseline_values
    else:
        base = _finite_array(baseline_values)
    if direction == "higher":
        return values - base
    return base - values

def bootstrap_ci(values, n_boot=10000, seed=42, statistic="mean"):
    values = _finite_array(values)
    rng = np.random.default_rng(seed)
    n = len(values)
    boot = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boot[i] = np.mean(sample) if statistic == "mean" else np.median(sample)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(lo), float(hi), boot

def holm_bonferroni(p_values):
    """Return Holm-Bonferroni adjusted p-values in original order."""
    p = np.asarray([np.nan if v is None else float(v) for v in p_values], dtype=np.float64)
    valid = np.isfinite(p)
    out = np.full_like(p, np.nan, dtype=np.float64)
    if valid.sum() == 0:
        return out
    idx = np.where(valid)[0]
    order = idx[np.argsort(p[valid])]
    m = len(order)
    adjusted_sorted = []
    for rank, original_i in enumerate(order):
        adjusted_sorted.append((m - rank) * p[original_i])
    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)
    for original_i, adj in zip(order, adjusted_sorted):
        out[original_i] = min(float(adj), 1.0)
    return out

def p_to_stars(p):
    if p is None or not np.isfinite(p):
        return "n/a"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def p_to_text(p):
    if p is None or not np.isfinite(p):
        return "n/a"
    if p < 1e-4:
        return "p < 0.0001"
    return f"p = {p:.4g}"

def load_optional_base_per_image():
    path = getattr(cfg, "BASELINE_PER_IMAGE_CSV", "")
    if path and os.path.exists(path):
        print("Using paired base-model per-image CSV:", path)
        return pd.read_csv(path)
    print("No paired base-model per-image CSV provided.")
    print("Proceeding with one-sample tests against reported base-paper UAVid mean values.")
    return None

base_per_image_df = load_optional_base_per_image()

def align_paired_data(prop_df, base_df, metric):
    """Align proposed and base per-image metric arrays by name/image_id/filename when possible."""
    if base_df is None or metric not in base_df.columns:
        return None, None

    prop = prop_df.copy()
    base = base_df.copy()

    prop_key = None
    base_key = None
    for k in ["name", "filename", "image_id", "path"]:
        if k in prop.columns:
            prop_key = k
            break
    for k in ["name", "filename", "image_id", "path"]:
        if k in base.columns:
            base_key = k
            break

    if prop_key is not None and base_key is not None:
        merged = prop[[prop_key, metric]].merge(
            base[[base_key, metric]],
            left_on=prop_key,
            right_on=base_key,
            suffixes=("_proposed", "_base")
        )
        if len(merged) > 0:
            return merged[f"{metric}_proposed"].astype(float).values, merged[f"{metric}_base"].astype(float).values

    if len(prop_df) == len(base_df):
        return prop_df[metric].astype(float).values, base_df[metric].astype(float).values

    return None, None

def one_sample_stats(values, base_mean, direction, metric, n_boot=10000, seed=42):
    vals = _finite_array(values)
    imp = _positive_improvement(vals, base_mean, direction)

    mean_val = float(np.mean(vals))
    std_val = float(np.std(vals, ddof=1))
    median_val = float(np.median(vals))
    mean_imp = float(np.mean(imp))
    median_imp = float(np.median(imp))
    ci_lo, ci_hi, _ = bootstrap_ci(imp, n_boot=n_boot, seed=seed, statistic="mean")
    med_lo, med_hi, _ = bootstrap_ci(imp, n_boot=n_boot, seed=seed + 1, statistic="median")

    n = len(imp)
    cohens_d = float(mean_imp / (np.std(imp, ddof=1) + 1e-12))
    hedges_g = float(cohens_d * (1.0 - (3.0 / (4.0 * n - 9.0)))) if n > 2 else cohens_d

    win_rate = float(np.mean(imp > 0.0))
    p_t = np.nan
    p_w = np.nan
    p_sign = np.nan

    if SCIPY_STATS_AVAILABLE:
        alternative = "greater" if direction == "higher" else "less"
        try:
            p_t = float(stats.ttest_1samp(vals, popmean=base_mean, alternative=alternative).pvalue)
        except Exception:
            p_t = np.nan
        try:
            p_w = float(stats.wilcoxon(imp, alternative="greater", zero_method="wilcox").pvalue)
        except Exception:
            p_w = np.nan
        try:
            wins = int(np.sum(imp > 0.0))
            trials = int(np.sum(imp != 0.0))
            p_sign = float(stats.binomtest(wins, trials, p=0.5, alternative="greater").pvalue) if trials > 0 else np.nan
        except Exception:
            p_sign = np.nan

    return {
        "comparison_type": "one_sample_vs_reported_base_mean",
        "metric": metric,
        "direction": direction,
        "n": len(vals),
        "base_reported_mean": float(base_mean),
        "proposed_mean": mean_val,
        "proposed_std": std_val,
        "proposed_median": median_val,
        "mean_improvement_positive_is_better": mean_imp,
        "median_improvement_positive_is_better": median_imp,
        "mean_bootstrap_95ci_low": ci_lo,
        "mean_bootstrap_95ci_high": ci_hi,
        "median_bootstrap_95ci_low": med_lo,
        "median_bootstrap_95ci_high": med_hi,
        "win_rate_positive_improvement": win_rate,
        "cohens_d": cohens_d,
        "hedges_g": hedges_g,
        "one_sample_t_p": p_t,
        "wilcoxon_signed_rank_p": p_w,
        "sign_test_p": p_sign,
    }

def paired_stats(proposed_values, base_values, direction, metric, n_boot=10000, seed=42):
    prop = _finite_array(proposed_values)
    base = _finite_array(base_values)
    n = min(len(prop), len(base))
    prop = prop[:n]
    base = base[:n]
    imp = _positive_improvement(prop, base, direction)

    mean_imp = float(np.mean(imp))
    median_imp = float(np.median(imp))
    ci_lo, ci_hi, _ = bootstrap_ci(imp, n_boot=n_boot, seed=seed, statistic="mean")
    med_lo, med_hi, _ = bootstrap_ci(imp, n_boot=n_boot, seed=seed + 1, statistic="median")

    cohens_dz = float(mean_imp / (np.std(imp, ddof=1) + 1e-12))
    n = len(imp)
    hedges_gz = float(cohens_dz * (1.0 - (3.0 / (4.0 * n - 9.0)))) if n > 2 else cohens_dz
    win_rate = float(np.mean(imp > 0.0))

    p_t = np.nan
    p_w = np.nan
    p_sign = np.nan
    if SCIPY_STATS_AVAILABLE:
        try:
            p_t = float(stats.ttest_rel(prop, base, alternative="greater" if direction == "higher" else "less").pvalue)
        except Exception:
            p_t = np.nan
        try:
            p_w = float(stats.wilcoxon(imp, alternative="greater", zero_method="wilcox").pvalue)
        except Exception:
            p_w = np.nan
        try:
            wins = int(np.sum(imp > 0.0))
            trials = int(np.sum(imp != 0.0))
            p_sign = float(stats.binomtest(wins, trials, p=0.5, alternative="greater").pvalue) if trials > 0 else np.nan
        except Exception:
            p_sign = np.nan

    return {
        "comparison_type": "paired_vs_base_model_same_split",
        "metric": metric,
        "direction": direction,
        "n": len(prop),
        "base_reported_mean": float(np.mean(base)),
        "proposed_mean": float(np.mean(prop)),
        "proposed_std": float(np.std(prop, ddof=1)),
        "proposed_median": float(np.median(prop)),
        "mean_improvement_positive_is_better": mean_imp,
        "median_improvement_positive_is_better": median_imp,
        "mean_bootstrap_95ci_low": ci_lo,
        "mean_bootstrap_95ci_high": ci_hi,
        "median_bootstrap_95ci_low": med_lo,
        "median_bootstrap_95ci_high": med_hi,
        "win_rate_positive_improvement": win_rate,
        "cohens_d": cohens_dz,
        "hedges_g": hedges_gz,
        "paired_t_p": p_t,
        "wilcoxon_signed_rank_p": p_w,
        "sign_test_p": p_sign,
    }

stat_rows = []
paired_available_any = False

for metric, info in BASELINE.items():
    if metric not in per_image_df.columns or not per_image_df[metric].notna().any():
        continue

    vals = per_image_df[metric].dropna().astype(float).values
    base_mean = float(info["value"])
    direction = info["direction"]

    prop_paired, base_paired = align_paired_data(per_image_df, base_per_image_df, metric)
    if prop_paired is not None and base_paired is not None and len(prop_paired) >= 2:
        paired_available_any = True
        row = paired_stats(prop_paired, base_paired, direction, metric, cfg.STAT_N_BOOT, cfg.SEED)
    else:
        row = one_sample_stats(vals, base_mean, direction, metric, cfg.STAT_N_BOOT, cfg.SEED)

    stat_rows.append(row)

stat_df = pd.DataFrame(stat_rows)

p_col = "wilcoxon_signed_rank_p" if "wilcoxon_signed_rank_p" in stat_df.columns else "one_sample_t_p"
stat_df["holm_adjusted_p"] = holm_bonferroni(stat_df[p_col].values)
stat_df["significance"] = stat_df["holm_adjusted_p"].apply(p_to_stars)
stat_df["statistically_better_0.05"] = (
    (stat_df["mean_bootstrap_95ci_low"] > 0.0) &
    (stat_df["holm_adjusted_p"] < 0.05)
)

stats_csv = os.path.join(cfg.WORK_DIR, "statistical_analysis_vs_basepaper_UAVid_C128.csv")
stat_df.to_csv(stats_csv, index=False)

print("Saved statistical table:", stats_csv)
display(stat_df)

# ============================================================
# Figure 1: distribution panel with base reported mean marker
# ============================================================
metrics_to_plot = [m for m in BASELINE.keys() if m in per_image_df.columns and per_image_df[m].notna().any()]
n_metrics = len(metrics_to_plot)
fig, axes = plt.subplots(1, n_metrics, figsize=(4.2 * n_metrics, 3.6), constrained_layout=True)
if n_metrics == 1:
    axes = [axes]

for ax, metric in zip(axes, metrics_to_plot):
    vals = _finite_array(per_image_df[metric].values)
    info = BASELINE[metric]
    ax.hist(vals, bins=35, alpha=0.75, edgecolor="black", linewidth=0.4)
    ax.axvline(info["value"], linestyle="--", linewidth=2, label="Base paper mean")
    ax.axvline(np.mean(vals), linestyle="-", linewidth=2, label="Proposed mean")
    ax.set_title(info["label"])
    ax.set_xlabel("Metric value")
    ax.set_ylabel("Image count")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

dist_fig = os.path.join(STAT_DIR, "stat_distribution_vs_base_mean_UAVid_C128.png")
fig.savefig(dist_fig, dpi=cfg.STAT_FIG_DPI, bbox_inches="tight")
plt.show()
print("Saved:", dist_fig)

# ============================================================
# Figure 2: improvement forest plot with bootstrap 95% CI
# Positive means proposed is better for every metric.
# ============================================================
plot_df = stat_df.copy()
y = np.arange(len(plot_df))
x = plot_df["mean_improvement_positive_is_better"].values
lo = plot_df["mean_bootstrap_95ci_low"].values
hi = plot_df["mean_bootstrap_95ci_high"].values
xerr = np.vstack([x - lo, hi - x])

fig, ax = plt.subplots(figsize=(8, 0.75 * len(plot_df) + 2.0))
ax.errorbar(x, y, xerr=xerr, fmt="o", capsize=5, linewidth=1.5)
ax.axvline(0.0, linestyle="--", linewidth=1.5)
ax.set_yticks(y)
ax.set_yticklabels([BASELINE[m]["label"] for m in plot_df["metric"]])
ax.set_xlabel("Mean improvement over base paper mean (positive = proposed better)")
ax.set_title("Bootstrap 95% CI of improvement")
ax.grid(True, axis="x", alpha=0.25)

for i, r in plot_df.iterrows():
    txt = f'{r["significance"]}, Holm {p_to_text(r["holm_adjusted_p"])}'
    ax.text(hi[i] + 0.02 * (abs(hi[i]) + 1e-9), i, txt, va="center", fontsize=9)

forest_fig = os.path.join(STAT_DIR, "stat_bootstrap_forest_plot_UAVid_C128.png")
fig.savefig(forest_fig, dpi=cfg.STAT_FIG_DPI, bbox_inches="tight")
plt.show()
print("Saved:", forest_fig)

# ============================================================
# Figure 3: ECDF of positive improvements
# ============================================================
fig, axes = plt.subplots(1, n_metrics, figsize=(4.2 * n_metrics, 3.6), constrained_layout=True)
if n_metrics == 1:
    axes = [axes]

for ax, metric in zip(axes, metrics_to_plot):
    vals = _finite_array(per_image_df[metric].values)
    info = BASELINE[metric]
    imp = _positive_improvement(vals, info["value"], info["direction"])
    imp_sorted = np.sort(imp)
    ecdf = np.arange(1, len(imp_sorted) + 1) / len(imp_sorted)
    ax.plot(imp_sorted, ecdf, linewidth=2)
    ax.axvline(0.0, linestyle="--", linewidth=1.5)
    ax.set_title(info["label"])
    ax.set_xlabel("Improvement over base mean")
    ax.set_ylabel("ECDF")
    ax.grid(True, alpha=0.25)

ecdf_fig = os.path.join(STAT_DIR, "stat_ecdf_improvement_UAVid_C128.png")
fig.savefig(ecdf_fig, dpi=cfg.STAT_FIG_DPI, bbox_inches="tight")
plt.show()
print("Saved:", ecdf_fig)

# ============================================================
# Figure 4: manuscript-style statistical table image
# ============================================================
table_cols = [
    "metric",
    "n",
    "base_reported_mean",
    "proposed_mean",
    "mean_improvement_positive_is_better",
    "mean_bootstrap_95ci_low",
    "mean_bootstrap_95ci_high",
    "win_rate_positive_improvement",
    "hedges_g",
    "holm_adjusted_p",
    "significance",
]
table_df = stat_df[table_cols].copy()
for c in table_df.columns:
    if c not in ["metric", "n", "significance"]:
        table_df[c] = table_df[c].map(lambda v: "n/a" if pd.isna(v) else f"{float(v):.6g}")

fig_height = 0.55 * len(table_df) + 1.6
fig, ax = plt.subplots(figsize=(14, fig_height))
ax.axis("off")
tbl = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    cellLoc="center",
    loc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.35)
ax.set_title("UAVid statistical superiority analysis, C=128", pad=14)

table_fig = os.path.join(STAT_DIR, "statistical_table_image_UAVid_C128.png")
fig.savefig(table_fig, dpi=cfg.STAT_FIG_DPI, bbox_inches="tight")
plt.show()
print("Saved:", table_fig)

stat_fig_manifest = pd.DataFrame({
    "figure": [
        "stat_distribution_vs_base_mean_UAVid_C128.png",
        "stat_bootstrap_forest_plot_UAVid_C128.png",
        "stat_ecdf_improvement_UAVid_C128.png",
        "statistical_table_image_UAVid_C128.png",
    ],
    "path": [dist_fig, forest_fig, ecdf_fig, table_fig]
})
manifest_csv = os.path.join(STAT_DIR, "statistical_figure_manifest_UAVid_C128.csv")
stat_fig_manifest.to_csv(manifest_csv, index=False)
print("Saved figure manifest:", manifest_csv)
display(stat_fig_manifest)


In [ ]:
# ============================================================
# 13C. Auto-generate manuscript-ready statistical statement
# ============================================================

def p_to_text_summary(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    if p < 1e-4:
        return "p < 0.0001"
    return f"p = {p:.4g}"

summary_lines = []
summary_lines.append("Statistical comparison on UAVid, C=128.")
summary_lines.append("Positive improvement means the proposed FAMamba-LSTMConvSR is better.")
summary_lines.append("")
summary_lines.append("Important note for publication:")
summary_lines.append("- If cfg.BASELINE_PER_IMAGE_CSV is empty, the analysis is a one-sample test against the base paper's reported mean values.")
summary_lines.append("- If cfg.BASELINE_PER_IMAGE_CSV points to base-model per-image results on the same split, the notebook automatically reports paired tests, which are stronger.")
summary_lines.append("")

for _, r in stat_df.iterrows():
    metric = r["metric"]
    direction = r["direction"]
    base = r["base_reported_mean"]
    mean = r["proposed_mean"]
    imp = r["mean_improvement_positive_is_better"]
    lo = r["mean_bootstrap_95ci_low"]
    hi = r["mean_bootstrap_95ci_high"]
    p_adj = r["holm_adjusted_p"]
    win = r["win_rate_positive_improvement"] * 100.0
    g = r["hedges_g"]
    sig = r["significance"]

    if direction == "higher":
        phr = f"{metric}: proposed mean {mean:.6f} vs base {base:.6f}, gain {imp:.6f}"
    else:
        phr = f"{metric}: proposed mean {mean:.6f} vs base {base:.6f}, reduction/improvement {imp:.6f}"

    phr += (
        f", bootstrap 95% CI [{lo:.6f}, {hi:.6f}], "
        f"Holm-adjusted {p_to_text_summary(p_adj)} ({sig}), "
        f"Hedges g = {g:.3f}, image-level superiority rate = {win:.2f}%."
    )
    summary_lines.append(phr)

all_sig = bool(stat_df["statistically_better_0.05"].all()) if len(stat_df) else False
summary_lines.append("")
if all_sig:
    summary_lines.append(
        "Conclusion: all tested metrics show statistically significant superiority after Holm correction "
        "and bootstrap confidence intervals remain above zero improvement."
    )
else:
    summary_lines.append(
        "Conclusion: at least one tested metric did not satisfy both Holm-corrected significance and positive bootstrap CI. "
        "Report the exact metric-level results rather than making an overgeneralized claim."
    )

summary_lines.append("")
summary_lines.append("Generated statistical figures:")
summary_lines.append(f"- {os.path.join(STAT_DIR, 'stat_distribution_vs_base_mean_UAVid_C128.png')}")
summary_lines.append(f"- {os.path.join(STAT_DIR, 'stat_bootstrap_forest_plot_UAVid_C128.png')}")
summary_lines.append(f"- {os.path.join(STAT_DIR, 'stat_ecdf_improvement_UAVid_C128.png')}")
summary_lines.append(f"- {os.path.join(STAT_DIR, 'statistical_table_image_UAVid_C128.png')}")

stat_txt = "\n".join(summary_lines)
stat_txt_path = os.path.join(cfg.WORK_DIR, "statistical_summary_text_UAVid_C128.txt")
with open(stat_txt_path, "w") as f:
    f.write(stat_txt)

print(stat_txt)
print("Saved:", stat_txt_path)


In [ ]:
# 14. Save sample visual outputs
# ============================================================

import matplotlib.pyplot as plt

@torch.no_grad()
def save_visual_samples(model, loader, out_dir, num_samples=8):
    os.makedirs(out_dir, exist_ok=True)
    model.eval()

    count = 0
    for lr, hr, names in loader:
        lr = lr.to(DEVICE)
        hr = hr.to(DEVICE)
        sr = torch.clamp(model(lr), 0, 1)
        if sr.shape[-2:] != hr.shape[-2:]:
            sr = F.interpolate(sr, size=hr.shape[-2:], mode="bicubic", align_corners=False)

        lr_up = F.interpolate(lr, size=hr.shape[-2:], mode="bicubic", align_corners=False)
        lr_up = torch.clamp(lr_up, 0, 1)

        for i in range(lr.size(0)):
            fig = plt.figure(figsize=(12, 4))

            imgs = [
                ("LR Bicubic", lr_up[i].detach().cpu()),
                ("SR", sr[i].detach().cpu()),
                ("HR", hr[i].detach().cpu())
            ]

            for j, (title, img) in enumerate(imgs):
                ax = fig.add_subplot(1, 3, j + 1)
                ax.imshow(img.permute(1, 2, 0).numpy())
                ax.set_title(title)
                ax.axis("off")

            save_path = os.path.join(out_dir, f"sample_{count:03d}_{names[i]}.png")
            plt.tight_layout()
            plt.savefig(save_path, dpi=150)
            plt.close(fig)

            count += 1
            if count >= num_samples:
                print(f"Saved {count} samples to {out_dir}")
                return

save_visual_samples(
    model,
    test_loader,
    out_dir=os.path.join(cfg.WORK_DIR, "visual_samples"),
    num_samples=8
)

In [ ]:
# ============================================================
# 15. Optional: LAM-style receptive-field visualization
# Fixed for cuDNN LSTM backward
# ============================================================

import matplotlib.pyplot as plt
import torch
import os

def lam_attribution(model, lr, patch_size=16):
    """
    lr: 1,3,H,W
    returns attribution map H,W

    Important:
    cuDNN RNN/LSTM backward requires train mode.
    So we temporarily use model.train() only for gradient attribution.
    """
    was_training = model.training

    # cuDNN LSTM backward needs train mode
    model.train()

    # Disable parameter gradients to save memory.
    # We only need gradient with respect to LR input.
    old_requires_grad = {}
    for name, p in model.named_parameters():
        old_requires_grad[name] = p.requires_grad
        p.requires_grad_(False)

    lr = lr.clone().detach().to(DEVICE)
    lr.requires_grad_(True)

    sr = model(lr)
    sr = torch.clamp(sr, 0, 1)

    _, _, h, w = sr.shape
    cy, cx = h // 2, w // 2
    ps = patch_size

    region = sr[:, :, cy - ps // 2:cy + ps // 2, cx - ps // 2:cx + ps // 2]
    score = region.mean()

    model.zero_grad(set_to_none=True)

    if lr.grad is not None:
        lr.grad.zero_()

    score.backward()

    grad = lr.grad.detach().abs().mean(dim=1)[0]
    grad = grad / (grad.sum() + 1e-12)

    # Restore parameter requires_grad
    for name, p in model.named_parameters():
        p.requires_grad_(old_requires_grad[name])

    # Restore original train/eval mode
    if was_training:
        model.train()
    else:
        model.eval()

    return grad.detach().cpu()


def diffusion_index(attr):
    """
    attr: H,W normalized attribution
    """
    h, w = attr.shape
    yy, xx = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")

    attr = attr / (attr.sum() + 1e-12)

    cy = (yy.float() * attr).sum()
    cx = (xx.float() * attr).sum()

    dist = torch.sqrt((yy.float() - cy) ** 2 + (xx.float() - cx) ** 2)
    return float((dist * attr).sum().item())


@torch.no_grad()
def get_one_sample(loader):
    for lr, hr, names in loader:
        return lr[:1], hr[:1], names[0]
    return None, None, None


lr_one, hr_one, name_one = get_one_sample(test_loader)

if lr_one is not None:
    attr = lam_attribution(model, lr_one, patch_size=16)
    di = diffusion_index(attr)

    plt.figure(figsize=(5, 5))
    plt.imshow(attr.numpy(), cmap="hot")
    plt.colorbar()
    plt.title(f"LAM-style attribution, DI={di:.4f}")
    plt.axis("off")

    lam_path = os.path.join(cfg.WORK_DIR, "lam_attribution.png")
    plt.savefig(lam_path, dpi=150, bbox_inches="tight")
    plt.close()

    print("Saved LAM-style attribution:", lam_path)
    print("Diffusion Index:", di)

print("All done.")

In [ ]:
# ============================================================
# Paper figure output folder
# ============================================================

import os
from pathlib import Path

FIG_DIR = os.path.join(cfg.WORK_DIR, "paper_figures")
os.makedirs(FIG_DIR, exist_ok=True)

print("Paper figures will be saved to:", FIG_DIR)

In [ ]:
# ============================================================
# Figure: qualitative SR comparison with zoom box
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn.functional as F
import numpy as np

def tensor_to_np_img(x):
    """
    x: C,H,W tensor in [0,1]
    """
    x = torch.clamp(x.detach().cpu(), 0, 1)
    return x.permute(1, 2, 0).numpy()

def draw_zoom_panel(ax, img_np, title, box=None, zoom_loc="upper left"):
    ax.imshow(img_np)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

    h, w = img_np.shape[:2]

    if box is None:
        # default central crop
        box_size = min(h, w) // 4
        x0 = w // 2 - box_size // 2
        y0 = h // 2 - box_size // 2
        box = (x0, y0, box_size, box_size)

    x0, y0, bw, bh = box

    rect = patches.Rectangle(
        (x0, y0), bw, bh,
        linewidth=1.5,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)

    # zoom crop inset
    crop = img_np[y0:y0+bh, x0:x0+bw]
    inset = ax.inset_axes([0.58, 0.58, 0.38, 0.38])
    inset.imshow(crop)
    inset.axis("off")
    for spine in inset.spines.values():
        spine.set_edgecolor("red")
        spine.set_linewidth(1.5)

def make_qualitative_figure(model, loader, save_path, num_samples=4):
    model.eval()
    saved = 0

    with torch.no_grad():
        for lr, hr, names in loader:
            lr = lr.to(DEVICE)
            hr = hr.to(DEVICE)

            sr = torch.clamp(model(lr), 0, 1)
            if sr.shape[-2:] != hr.shape[-2:]:
                sr = F.interpolate(sr, size=hr.shape[-2:], mode="bicubic", align_corners=False)

            bicubic = F.interpolate(
                lr,
                size=hr.shape[-2:],
                mode="bicubic",
                align_corners=False
            )
            bicubic = torch.clamp(bicubic, 0, 1)

            for i in range(lr.size(0)):
                y_sr = rgb_to_y(sr[i:i+1])
                y_hr = rgb_to_y(hr[i:i+1])
                y_bi = rgb_to_y(bicubic[i:i+1])

                psnr_sr = calc_psnr(y_sr, y_hr).item()
                ssim_sr = calc_ssim(y_sr, y_hr).item()
                psnr_bi = calc_psnr(y_bi, y_hr).item()
                ssim_bi = calc_ssim(y_bi, y_hr).item()

                hr_np = tensor_to_np_img(hr[i])
                bi_np = tensor_to_np_img(bicubic[i])
                sr_np = tensor_to_np_img(sr[i])

                h, w = hr_np.shape[:2]
                box_size = min(h, w) // 4

                # You can manually change this box for better visual regions
                x0 = w // 2 - box_size // 2
                y0 = h // 2 - box_size // 2
                box = (x0, y0, box_size, box_size)

                fig, axes = plt.subplots(1, 3, figsize=(12, 4))

                draw_zoom_panel(
                    axes[0],
                    bi_np,
                    f"Bicubic\n{psnr_bi:.2f}/{ssim_bi:.4f}",
                    box=box
                )

                draw_zoom_panel(
                    axes[1],
                    sr_np,
                    f"FAMamba-LSTMConvSR\n{psnr_sr:.2f}/{ssim_sr:.4f}",
                    box=box
                )

                draw_zoom_panel(
                    axes[2],
                    hr_np,
                    "HR\n∞/1",
                    box=box
                )

                plt.tight_layout()

                out = save_path.replace(".png", f"_{saved:02d}_{names[i]}.png")
                plt.savefig(out, dpi=300, bbox_inches="tight")
                plt.close(fig)

                print("Saved:", out)

                saved += 1
                if saved >= num_samples:
                    return

qual_path = os.path.join(FIG_DIR, "qualitative_comparison.png")
make_qualitative_figure(model, test_loader, qual_path, num_samples=6)

In [ ]:
# ============================================================
# Figure: feature map evolution
# ViLG → CNN → Mamba → Frequency → Fusion
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch

def get_core_model(model):
    return model.module if hasattr(model, "module") else model

def normalize_feature_map(fmap):
    """
    fmap: H,W tensor
    """
    fmap = fmap.detach().float().cpu()
    fmap = fmap - fmap.min()
    fmap = fmap / (fmap.max() + 1e-8)
    return fmap.numpy()

def save_feature_evolution(model, loader, save_path, channel_ids=[0, 8, 16]):
    core = get_core_model(model)
    model.eval()

    # Take first test sample
    lr, hr, names = next(iter(loader))
    lr = lr[:1].to(DEVICE)
    hr = hr[:1].to(DEVICE)

    activations = {}

    def hook_fn(name):
        def _hook(module, inp, out):
            if isinstance(out, tuple):
                out = out[0]
            activations[name] = out.detach()
        return _hook

    # We hook the first HybridNeuralBlock
    # block[0] contains: vilg, famcrg, fusion
    handles = []

    try:
        block0 = core.blocks[0]
        handles.append(block0.vilg.register_forward_hook(hook_fn("ViLG")))

        # first FAMCB inside FAMCRG
        famcb0 = block0.famcrg.blocks[0]
        handles.append(famcb0.cnn.register_forward_hook(hook_fn("CNN Branch")))
        handles.append(famcb0.mamba.register_forward_hook(hook_fn("Mamba Branch")))
        handles.append(famcb0.freq.register_forward_hook(hook_fn("Frequency Branch")))
        handles.append(block0.fusion.register_forward_hook(hook_fn("Gated Fusion")))

    except Exception as e:
        print("[WARN] Hook setup failed:", e)
        return

    with torch.no_grad():
        sr = torch.clamp(model(lr), 0, 1)

    for h in handles:
        h.remove()

    # Prepare visual rows
    stages = ["ViLG", "CNN Branch", "Mamba Branch", "Frequency Branch", "Gated Fusion"]
    stages = [s for s in stages if s in activations]

    nrows = len(channel_ids)
    ncols = len(stages)

    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))

    if nrows == 1:
        axes = np.expand_dims(axes, axis=0)
    if ncols == 1:
        axes = np.expand_dims(axes, axis=1)

    for r, ch in enumerate(channel_ids):
        for c, stage in enumerate(stages):
            fmap = activations[stage][0]  # C,H,W
            ch_idx = min(ch, fmap.shape[0] - 1)
            fmap_np = normalize_feature_map(fmap[ch_idx])

            axes[r, c].imshow(fmap_np, cmap="viridis")
            axes[r, c].axis("off")
            if r == 0:
                axes[r, c].set_title(stage, fontsize=10)
            if c == 0:
                axes[r, c].set_ylabel(f"Channel {ch_idx}", fontsize=10)

    plt.suptitle("Feature Map Evolution in FAMamba-LSTMConvSR", fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved feature evolution figure:", save_path)

feature_path = os.path.join(FIG_DIR, "feature_evolution_famamba.png")
save_feature_evolution(model, test_loader, feature_path, channel_ids=[0, 8, 16])

In [ ]:
# ============================================================
# Figure: LAM-style receptive-field visualization
# Fixed for cuDNN LSTM backward
# ============================================================

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import os

def lam_attribution_paper(model, lr, patch_size=16):
    """
    LAM-style gradient attribution.
    Fixed: cuDNN LSTM backward requires train mode.
    """
    was_training = model.training

    # cuDNN RNN/LSTM backward requires train mode
    model.train()

    # Freeze model parameters; we only need gradient w.r.t. LR input.
    old_requires_grad = {}
    for name, p in model.named_parameters():
        old_requires_grad[name] = p.requires_grad
        p.requires_grad_(False)

    lr = lr.clone().detach().to(DEVICE)
    lr.requires_grad_(True)

    sr = model(lr)
    sr = torch.clamp(sr, 0, 1)

    _, _, h, w = sr.shape
    cy, cx = h // 2, w // 2
    ps = patch_size

    region = sr[:, :, cy - ps // 2:cy + ps // 2, cx - ps // 2:cx + ps // 2]
    score = region.mean()

    model.zero_grad(set_to_none=True)

    if lr.grad is not None:
        lr.grad.zero_()

    score.backward()

    attr = lr.grad.detach().abs().mean(dim=1)[0]
    attr = attr / (attr.sum() + 1e-12)

    # Restore parameter gradients
    for name, p in model.named_parameters():
        p.requires_grad_(old_requires_grad[name])

    # Restore model mode
    if was_training:
        model.train()
    else:
        model.eval()

    return attr.detach().cpu(), sr.detach()


def diffusion_index(attr):
    h, w = attr.shape
    yy, xx = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")

    attr = attr / (attr.sum() + 1e-12)

    cy = (yy.float() * attr).sum()
    cx = (xx.float() * attr).sum()

    dist = torch.sqrt((yy.float() - cy) ** 2 + (xx.float() - cx) ** 2)
    return float((dist * attr).sum().item())


def save_lam_paper_figure(model, loader, save_path):
    lr, hr, names = next(iter(loader))
    lr = lr[:1].to(DEVICE)
    hr = hr[:1].to(DEVICE)

    attr, sr = lam_attribution_paper(model, lr, patch_size=16)
    di = diffusion_index(attr)

    with torch.no_grad():
        bicubic = F.interpolate(
            lr,
            size=hr.shape[-2:],
            mode="bicubic",
            align_corners=False
        )
        bicubic = torch.clamp(bicubic, 0, 1)
        sr = torch.clamp(sr, 0, 1)

    lr_up_np = tensor_to_np_img(bicubic[0])
    sr_np = tensor_to_np_img(sr[0])
    hr_np = tensor_to_np_img(hr[0])
    attr_np = attr.numpy()

    fig, axes = plt.subplots(1, 4, figsize=(14, 4))

    axes[0].imshow(lr_up_np)
    axes[0].set_title("LR Bicubic")
    axes[0].axis("off")

    axes[1].imshow(hr_np)
    axes[1].set_title("HR")
    axes[1].axis("off")

    axes[2].imshow(sr_np)
    axes[2].set_title("FAMamba-LSTMConvSR")
    axes[2].axis("off")

    im = axes[3].imshow(attr_np, cmap="hot")
    axes[3].set_title(f"LAM Attribution\nDI={di:.4f}")
    axes[3].axis("off")
    fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved LAM paper figure:", save_path)
    print("Diffusion Index:", di)


lam_paper_path = os.path.join(FIG_DIR, "lam_paper_famamba.png")
save_lam_paper_figure(model, test_loader, lam_paper_path)

In [ ]:
# ============================================================
# Figure: proposed FAMamba-LSTMConvSR architecture diagram
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_box(ax, xy, w, h, text, fc="#E8F1FF", ec="#2B5FAB", fontsize=9):
    box = patches.FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=1.5,
        edgecolor=ec,
        facecolor=fc
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + w / 2,
        xy[1] + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        wrap=True
    )

def arrow(ax, start, end):
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops=dict(arrowstyle="->", lw=1.6, color="black")
    )

def save_architecture_diagram(save_path):
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 6)
    ax.axis("off")

    # Main pipeline
    draw_box(ax, (0.3, 2.5), 1.1, 0.8, "LR\nImage", "#FFF4E6", "#C27000")
    draw_box(ax, (1.8, 2.5), 1.3, 0.8, "Shallow\nConv", "#E8F1FF", "#2B5FAB")
    draw_box(ax, (3.5, 2.0), 4.2, 1.8, "Hybrid Neural Blocks × K\n\nViLG + FAMCRG + Gated Fusion", "#EFFFF2", "#258A3C")
    draw_box(ax, (8.2, 2.5), 1.5, 0.8, "Global\nResidual", "#F3E8FF", "#7A3DB8")
    draw_box(ax, (10.2, 2.5), 1.4, 0.8, "PixelShuffle\n×4", "#FFF4E6", "#C27000")
    draw_box(ax, (12.1, 2.5), 1.2, 0.8, "SR\nImage", "#FFECEC", "#C43B3B")

    arrow(ax, (1.4, 2.9), (1.8, 2.9))
    arrow(ax, (3.1, 2.9), (3.5, 2.9))
    arrow(ax, (7.7, 2.9), (8.2, 2.9))
    arrow(ax, (9.7, 2.9), (10.2, 2.9))
    arrow(ax, (11.6, 2.9), (12.1, 2.9))

    # Internal module details
    draw_box(ax, (3.8, 0.6), 1.5, 0.8, "ViLG\nLSTM Guidance", "#E8F1FF", "#2B5FAB")
    draw_box(ax, (5.7, 0.6), 1.8, 0.8, "FAMCRG\nHybrid Refocus", "#EFFFF2", "#258A3C")
    draw_box(ax, (7.9, 0.6), 1.8, 0.8, "Gated Top-Down\nFusion", "#F3E8FF", "#7A3DB8")

    arrow(ax, (5.3, 1.0), (5.7, 1.0))
    arrow(ax, (7.5, 1.0), (7.9, 1.0))

    # FAMCRG branches
    draw_box(ax, (5.0, 4.6), 1.4, 0.7, "CNN Branch\nLocal Texture", "#FFFFFF", "#444444")
    draw_box(ax, (6.7, 4.6), 1.4, 0.7, "Mamba Branch\nLong Range", "#FFFFFF", "#444444")
    draw_box(ax, (8.4, 4.6), 1.6, 0.7, "Wavelet Branch\nHigh Frequency", "#FFFFFF", "#444444")
    draw_box(ax, (10.3, 4.6), 1.4, 0.7, "Branch-wise\nGating", "#FFFFFF", "#444444")

    arrow(ax, (6.4, 4.95), (6.7, 4.95))
    arrow(ax, (8.1, 4.95), (8.4, 4.95))
    arrow(ax, (10.0, 4.95), (10.3, 4.95))

    ax.text(
        7.5, 5.65,
        "Proposed Frequency-Aware Mamba–CNN Block (FAMCB)",
        fontsize=13,
        ha="center",
        fontweight="bold"
    )

    ax.text(
        7.0, 0.1,
        "FAMamba-LSTMConvSR: Vision-LSTM guidance + CNN local refinement + Mamba long-range modeling + wavelet high-frequency enhancement",
        fontsize=10,
        ha="center"
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved architecture diagram:", save_path)

arch_path = os.path.join(FIG_DIR, "architecture_famamba_lstmconvsr.png")
save_architecture_diagram(arch_path)

In [ ]:
# ============================================================
# Figure: metrics bar chart for current run
# ============================================================

import matplotlib.pyplot as plt
import os

def save_metric_bar_chart(test_stats, save_path):
    metrics = ["psnr_y", "ssim_y", "ms_ssim_rgb", "ergas", "gmsd", "lpips", "dists", "clipiqa"]
    values = [test_stats.get(m, None) for m in metrics]

    metrics_clean = []
    values_clean = []
    for m, v in zip(metrics, values):
        if v is not None:
            metrics_clean.append(m)
            values_clean.append(v)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(metrics_clean, values_clean)
    ax.set_title("Evaluation Metrics of FAMamba-LSTMConvSR")
    ax.set_ylabel("Metric Value")
    ax.tick_params(axis="x", rotation=30)

    for i, v in enumerate(values_clean):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved metric chart:", save_path)

metric_path = os.path.join(FIG_DIR, "metric_bar_chart.png")
save_metric_bar_chart(test_stats, metric_path)

In [ ]:
# ============================================================
# FINAL CELL: ZIP ALL UAVid RESULTS + DOWNLOAD LINK FIXED FOR KAGGLE
# ============================================================

import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display, HTML

# Use cfg.WORK_DIR if available; otherwise use the default UAVid C=128 folder
try:
    RESULT_DIR = Path(cfg.WORK_DIR)
except NameError:
    RESULT_DIR = Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_exact_stat")

# Final zip name and path
ZIP_NAME = "famamba_lstmconvsr_uavid_C128_ALL_RESULTS.zip"
ZIP_PATH = Path("/kaggle/working") / ZIP_NAME

# Optional extra files to include if available
EXTRA_PATHS = [
    Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_exact_stats_UPDATED.ipynb"),
    Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_exact_stats_cell_by_cell.py"),
    Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_full_results_and_statistics.zip"),
]

print("RESULT_DIR:", RESULT_DIR)
print("RESULT_DIR exists:", RESULT_DIR.exists())

if not RESULT_DIR.exists():
    print("\nAvailable folders/files in /kaggle/working:")
    for p in sorted(Path("/kaggle/working").iterdir()):
        print(" -", p)
    raise FileNotFoundError(
        f"Result directory not found: {RESULT_DIR}\n"
        "Check cfg.WORK_DIR or the actual output folder name above."
    )

# Remove old zip
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

def add_path_to_zip(zipf, path: Path, arc_prefix: str = ""):
    if not path.exists():
        return

    if path.is_file():
        if path.resolve() == ZIP_PATH.resolve():
            return
        arcname = Path(arc_prefix) / path.name if arc_prefix else path.name
        zipf.write(path, arcname=str(arcname))

    elif path.is_dir():
        for file in path.rglob("*"):
            if file.is_file():
                if file.resolve() == ZIP_PATH.resolve():
                    continue
                arcname = Path(arc_prefix) / file.relative_to(path)
                zipf.write(file, arcname=str(arcname))

# Create zip
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
    add_path_to_zip(zipf, RESULT_DIR, arc_prefix=RESULT_DIR.name)

    for extra in EXTRA_PATHS:
        if extra.exists():
            add_path_to_zip(zipf, extra, arc_prefix="extra_files")

# Verify
if not ZIP_PATH.exists():
    raise RuntimeError("ZIP was not created.")

zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)

print("=" * 80)
print("ALL UAVid C=128 RESULTS ZIPPED SUCCESSFULLY")
print("=" * 80)
print("ZIP path:", ZIP_PATH)
print(f"ZIP size: {zip_size_mb:.2f} MB")
print("=" * 80)

# IMPORTANT: Kaggle FileLink works better with relative paths from /kaggle/working
os.chdir("/kaggle/working")

print("\nDownload link:")
display(FileLink(ZIP_NAME))

# Backup clickable HTML link
display(HTML(f'<a href="{ZIP_NAME}" download>Download {ZIP_NAME}</a>'))

# Also show files in output directory for manual download
print("\nFiles in /kaggle/working:")
for p in sorted(Path("/kaggle/working").glob("*")):
    size = p.stat().st_size / (1024 * 1024) if p.is_file() else 0
    print(f" - {p.name} {'(' + str(round(size, 2)) + ' MB)' if p.is_file() else '[folder]'}")

In [ ]:
# ============================================================
# FINAL CELL: ZIP ALL UAVid RESULTS AND CREATE DIRECT DOWNLOAD LINK
# ============================================================

import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

# Main UAVid result directory used in the C=128 notebook
RESULT_DIR = Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_exact_stat")

# Final ZIP output path
ZIP_PATH = Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_ALL_RESULTS.zip")

# Optional extra files/folders to include if they exist
EXTRA_PATHS = [
    Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_exact_stats_UPDATED.ipynb"),
    Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_exact_stats_cell_by_cell.py"),
    Path("/kaggle/working/famamba_lstmconvsr_uavid_C128_full_results_and_statistics.zip"),
]

# Remove old zip if it exists
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

if not RESULT_DIR.exists():
    raise FileNotFoundError(
        f"Result directory not found: {RESULT_DIR}\n"
        "Check that cfg.WORK_DIR matches this path and that the notebook has finished running."
    )

def add_path_to_zip(zipf, path: Path, arc_prefix: str = ""):
    """Add a file or folder to zip safely."""
    if not path.exists():
        return

    if path.is_file():
        if path.resolve() == ZIP_PATH.resolve():
            return
        arcname = Path(arc_prefix) / path.name if arc_prefix else path.name
        zipf.write(path, arcname=str(arcname))

    elif path.is_dir():
        for file in path.rglob("*"):
            if file.is_file():
                if file.resolve() == ZIP_PATH.resolve():
                    continue
                arcname = Path(arc_prefix) / file.relative_to(path)
                zipf.write(file, arcname=str(arcname))

# Create ZIP
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
    # Add all main UAVid result files
    add_path_to_zip(zipf, RESULT_DIR, arc_prefix=RESULT_DIR.name)

    # Add optional extra files if present
    for extra in EXTRA_PATHS:
        if extra.exists():
            add_path_to_zip(zipf, extra, arc_prefix="extra_files")

# Print ZIP summary
zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)

print("=" * 80)
print("ALL UAVid RESULTS ZIPPED SUCCESSFULLY")
print("=" * 80)
print(f"ZIP file: {ZIP_PATH}")
print(f"ZIP size: {zip_size_mb:.2f} MB")
print("=" * 80)

# Direct download link in Kaggle notebook output
display(FileLink(str(ZIP_PATH)))